# TIER.0) Load scored universe (single source of truth)

In [ ]:
# ============================================================
# TIER.0) Load scored universe (single source of truth)
# - Input: artifacts/eval_universe/eval_scored_DG_V3.parquet
# - Output: eval_base (row-level grain: NPI x HCPCS x POS x Year)
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

EVAL_PATH = Path("artifacts/eval_universe/eval_scored_DG_V3.parquet")
assert EVAL_PATH.exists(), f"Missing: {EVAL_PATH}"

eval_base = pd.read_parquet(EVAL_PATH)

print("Loaded eval_base:", eval_base.shape)
print("Columns:", len(eval_base.columns))

# Basic required columns for tiering
REQ = [
    "row_id",
    "Rndrng_NPI", "provider_type", "state",
    "HCPCS_Cd", "Year",
    "Place_Of_Srvc",
    "observed_cost", "expected_cost", "residual", "oe_ratio", "log_oe",
    "services", "benes",
    "expected_cost_support_tier",
    "high_confidence_anomaly_candidate",
]
missing = [c for c in REQ if c not in eval_base.columns]
if missing:
    raise KeyError(f"eval_scored_DG_V3 missing required columns for tiering: {missing}")

# Optional sanity on grain (should be unique at (NPI,HCPCS,POS,Year))
keys = ["Rndrng_NPI","HCPCS_Cd","Place_Of_Srvc","Year"]
n_dupe = int(eval_base.duplicated(keys).sum())
print("Duplicate key rows:", n_dupe)
if n_dupe:
    display(eval_base.loc[eval_base.duplicated(keys, keep=False), keys + ["row_id"]].head(20))

# -----------------------------
# TIER.0 sanity footer (quick distribution + NaN checks)
# -----------------------------

# Coerce to numeric safely (won't modify other cols)
for c in ["observed_cost", "expected_cost", "residual", "oe_ratio", "log_oe", "services", "benes"]:
    if c in eval_base.columns:
        eval_base[c] = pd.to_numeric(eval_base[c], errors="coerce")

def _quick_stats(df: pd.DataFrame, col: str) -> None:
    s = df[col]
    n = len(s)
    n_na = int(s.isna().sum())
    n_inf = int(np.isinf(s.to_numpy(dtype="float64", copy=False)).sum()) if np.issubdtype(s.dtype, np.number) else 0

    finite = s[np.isfinite(s.to_numpy(dtype="float64", copy=False))]
    if len(finite) == 0:
        print(f"{col}: all NaN/inf (n={n:,}, n_na={n_na:,}, n_inf={n_inf:,})")
        return

    qs = np.nanpercentile(finite.to_numpy(dtype="float64"), [0, 1, 5, 50, 95, 99, 100])
    print(
        f"{col}: n={n:,} | n_na={n_na:,} ({n_na/n*100:.3f}%) | n_inf={n_inf:,} | "
        f"min={qs[0]:.6g} p01={qs[1]:.6g} p05={qs[2]:.6g} median={qs[3]:.6g} "
        f"p95={qs[4]:.6g} p99={qs[5]:.6g} max={qs[6]:.6g}"
    )

print("\nTIER.0 sanity footer (key scored columns)")
for col in ["expected_cost", "residual", "log_oe", "oe_ratio", "observed_cost"]:
    _quick_stats(eval_base, col)

# Extra: quickly spot pathological expected_cost near 0 (can blow up oe_ratio)
if "expected_cost" in eval_base.columns:
    eps_levels = [1, 1e-1, 1e-2, 1e-3, 1e-4, 1e-6]
    print("\nexpected_cost near-zero counts:")
    for eps in eps_levels:
        cnt = int((eval_base["expected_cost"].abs() < eps).sum())
        print(f"  |expected_cost| < {eps:g}: {cnt:,} ({cnt/len(eval_base)*100:.3f}%)")

# ============================================================
# TIER.0 sanity footer (oe_ratio extreme tail counts)
# Add this at the very bottom of TIER.0
# ============================================================

# assumes eval_base exists (our loaded scored universe)
if "eval_base" not in globals():
    raise NameError("Missing eval_base. Run the TIER.0 load cell first.")

if "oe_ratio" not in eval_base.columns:
    raise KeyError("eval_base missing oe_ratio.")

s = pd.to_numeric(eval_base["oe_ratio"], errors="coerce")
n = len(s)
n_na = int(s.isna().sum())
finite = s[np.isfinite(s.to_numpy())]

thresholds = [10, 50, 100, 1000]
print("\nTIER.0 sanity footer (oe_ratio extreme tail counts)")
print(f"oe_ratio: n={n:,} | n_na={n_na:,} ({(n_na/n*100):.3f}%) | n_finite={len(finite):,}")

for t in thresholds:
    k = int((finite > t).sum())
    pct = (k / len(finite) * 100) if len(finite) else float("nan")
    print(f"  oe_ratio > {t:>4}: {k:,} ({pct:.4f}%)")

# Optional: show the worst 10 for quick inspection (toggle)
SHOW_TOP = False
if SHOW_TOP:
    tmp = eval_base.loc[np.isfinite(s.to_numpy()), ["row_id","Rndrng_NPI","HCPCS_Cd","Year","Place_Of_Srvc","observed_cost","expected_cost","residual","oe_ratio","log_oe"]].copy()
    tmp["oe_ratio"] = pd.to_numeric(tmp["oe_ratio"], errors="coerce")
    display(tmp.sort_values("oe_ratio", ascending=False).head(10))

## Explain why `oe_ratio` explodes: is it due to tiny `expected_cost` for tiny `observed_cost` or tiny `expected_cost` for material dollar `observed_cost`

In [ ]:
# ------------------------------------------------------------
# TIER.0 add-on: diagnose what drives extreme oe_ratio tails
# ------------------------------------------------------------
tmp = eval_base.copy()

tmp["oe_ratio"] = pd.to_numeric(tmp["oe_ratio"], errors="coerce")
tmp["expected_cost"] = pd.to_numeric(tmp["expected_cost"], errors="coerce")
tmp["observed_cost"] = pd.to_numeric(tmp["observed_cost"], errors="coerce")

finite = tmp[np.isfinite(tmp["oe_ratio"])].copy()

for t in [10, 50, 100, 1000]:
    tail = finite[finite["oe_ratio"] > t]
    if len(tail) == 0:
        continue
    print(f"\nTail oe_ratio > {t} (n={len(tail):,})")
    print("  expected_cost median / p05 / p01:",
          float(tail["expected_cost"].median()),
          float(tail["expected_cost"].quantile(0.05)),
          float(tail["expected_cost"].quantile(0.01)))
    print("  observed_cost median / p05 / p01:",
          float(tail["observed_cost"].median()),
          float(tail["observed_cost"].quantile(0.05)),
          float(tail["observed_cost"].quantile(0.01)))
    print("  share expected_cost < 1:", float((tail["expected_cost"] < 1).mean() * 100), "%")
    print("  share expected_cost < 0.1:", float((tail["expected_cost"] < 0.1).mean() * 100), "%")

## `oe_ratio > 10`, `oe_ratio > 50` and `oe_ratio > 100` have a different reason than `oe_ratio > 1000`

In [ ]:
# TIER.0 add-on: Quantify how many of these tail rows are actually “material dollars”

for t in [10, 50, 100, 1000]:
      tail = finite[finite["oe_ratio"] > t]
      if len(tail) == 0:
            continue
      print(f"\nTail oe_ratio > {t} (n={len(tail):,})")
      print("  share observed_cost > $10:", float((tail["observed_cost"] > 10).mean() * 100), "%")
      print("  share residual > $10:", float((tail["residual"] > 10).mean() * 100), "%")

> `oe_ratio > 10`, `oe_ratio > 50` and `oe_ratio > 100` happen when the `expected_cost` is tiny but `observed_cost` is material dollars, while `oe_ratio > 1000` happen when `expected_cost` is tiny but the `obsereved_cost` is also small (mostly under $10)

In [ ]:
# TIER.0 add-on: Make the “two regimes” visible in one line per threshold

for t in [10, 50, 100, 1000]:
    tail = finite[finite["oe_ratio"] > t]
    if len(tail) == 0:
        continue
    print(f"\nTail oe_ratio > {t} (n={len(tail):,})")
    print("  expected_cost median:", float(tail["expected_cost"].median()))
    print("  expected_cost p01:", float(tail["expected_cost"].quantile(0.01)))
    print("  observed_cost median:", float(tail["observed_cost"].median()))

# TIER.1) Minimal anomaly feature layer for tiering

In [ ]:
# ============================================================
# TIER.1) Minimal anomaly feature layer for tiering
# Builds:
#   - is_high_conf
#   - log_oe_pct_in_slice within (HCPCS_Cd, Year)
#   - slice_n within (HCPCS_Cd, Year)
# Notes:
#   - Keeps definitions aligned with anomaly surfacing notebook.
# ============================================================

import numpy as np
import pandas as pd

df = eval_base.copy()

# -----------------------------
# High-confidence definition (same knobs as ANOM.1)
# -----------------------------
MIN_SERVICES = 50
MIN_BENES = 20
HIGH_CONF_TIERS = {"high", "medium_high"}

df["is_high_conf"] = (
    df["expected_cost_support_tier"].astype(str).isin(HIGH_CONF_TIERS)
    & (df["services"].fillna(0) >= MIN_SERVICES)
    & (df["benes"].fillna(0) >= MIN_BENES)
)

# Keep existing candidate flag (force bool)
df["high_confidence_anomaly_candidate"] = df["high_confidence_anomaly_candidate"].astype(bool)

# -----------------------------
# Slice features: (HCPCS_Cd, Year)
# -----------------------------
slice_cols = ["HCPCS_Cd", "Year"]

def _pct_rank(s: pd.Series) -> pd.Series:
    return s.rank(pct=True, method="average")

df["log_oe"] = pd.to_numeric(df["log_oe"], errors="coerce")
df["log_oe_pct_in_slice"] = df.groupby(slice_cols, dropna=False)["log_oe"].transform(_pct_rank)
df["slice_n"] = df.groupby(slice_cols, dropna=False)["row_id"].transform("size")

eval_tier = df
print("Defined eval_tier:", eval_tier.shape)

# Quick sanity
print("is_high_conf true %:", float(eval_tier["is_high_conf"].mean() * 100))
print("slice_n min/median/p1/p99/max:",
      int(eval_tier["slice_n"].min()),
      float(eval_tier["slice_n"].median()),
      float(eval_tier["slice_n"].quantile(0.01)),
      float(eval_tier["slice_n"].quantile(0.99)),
      int(eval_tier["slice_n"].max()))

# How many rows have slice_n >= 50? That becomes our standard validity gate downstream
print((eval_tier["slice_n"] >= 50).value_counts())

# TIER.2) Provider summaries (robust repeat offenders + shock severity), *NO `TOP_N` trimming*

In [ ]:
# ============================================================
# TIER.2) Provider summaries (robust repeat offenders + shock severity), NO TOP_N trimming
# Produces:
#   - provider_summary_robust_sizeaware_all
#   - provider_summary_mag_sizeaware_all
# Notes:
#   - Uses same event definitions as ANOM.3.a.1 and ANOM.3.b.1
#   - Keeps provider grain: (Rndrng_NPI, provider_type, state)
# ============================================================

import numpy as np
import pandas as pd

df = eval_tier.copy()

# -----------------------------
# Shared knobs (tiering)
# -----------------------------
MIN_SLICE_N = 50

USE_MIN_EXPECTED_FILTER = True
MIN_EXPECTED = 1e-2

# For shock severity cutoff
LOG_OE_Q = 0.995

# Ensure numeric
for c in ["log_oe","residual","expected_cost","services","benes"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# Optional denom hygiene (recommended)
if USE_MIN_EXPECTED_FILTER:
    df = df[df["expected_cost"].notna() & (df["expected_cost"] >= MIN_EXPECTED)].copy()

# -----------------------------
# ROBUST extreme-event flag (repeat offenders): ANOM.3.a.1 definition
# -----------------------------
df["is_row_anomalous_robust_sizeaware"] = (
    (df["log_oe"] > 0)
    & (df["residual"] > 0)
    & (df["is_high_conf"])
    & (df["log_oe_pct_in_slice"] >= 0.99)
    & (df["slice_n"] >= MIN_SLICE_N)
)

# Provider summary (robust)
GROUP = ["Rndrng_NPI","provider_type","state"]
provider_summary_robust_sizeaware_all = (
    df.groupby(GROUP, dropna=False)
      .agg(
          n_rows=("row_id","size"),
          n_anom_rows_robust=("is_row_anomalous_robust_sizeaware","sum"),
          anom_rate_pct_robust=("is_row_anomalous_robust_sizeaware", lambda s: float(s.mean()*100)),
          n_unique_codes=("HCPCS_Cd", pd.Series.nunique),
          n_unique_years=("Year", pd.Series.nunique),
          median_log_oe=("log_oe","median"),
          median_residual=("residual","median"),
          p75_log_oe=("log_oe", lambda s: float(np.nanpercentile(s, 75))),
          p90_log_oe=("log_oe", lambda s: float(np.nanpercentile(s, 90))),
          p75_residual=("residual", lambda s: float(np.nanpercentile(s, 75))),
          p90_residual=("residual", lambda s: float(np.nanpercentile(s, 90))),
          total_services=("services","sum"),
          total_benes=("benes","sum"),
          pct_high_conf_rows=("is_high_conf", lambda s: float(s.mean()*100)),
      )
      .reset_index()
)

# Optional: keep providers with >0 events as a field, but do NOT filter here by default
provider_summary_robust_sizeaware_all["provider_anom_score_robust"] = (
    provider_summary_robust_sizeaware_all["n_anom_rows_robust"]
    + 0.25 * provider_summary_robust_sizeaware_all["n_unique_codes"]
    + 0.25 * provider_summary_robust_sizeaware_all["n_unique_years"]
)

print("provider_summary_robust_sizeaware_all:", provider_summary_robust_sizeaware_all.shape)

# -----------------------------
# MAG shock-event flag: ANOM.3.b.1 definition
# - severity cutoff computed on BASE (log_oe>0 & residual>0) after denom hygiene
# -----------------------------
BASE = df[(df["log_oe"] > 0) & (df["residual"] > 0)].copy()
if len(BASE) == 0:
    raise ValueError("No BASE rows found for severity cut (log_oe>0 & residual>0). Check inputs/filters.")

log_oe_severity_cut = float(BASE["log_oe"].quantile(LOG_OE_Q))
print(f"log_oe_severity_cut (q={LOG_OE_Q}): {log_oe_severity_cut:.6f} (oe_ratio ~ {np.exp(log_oe_severity_cut):.2f}x)")

df["is_row_anomalous_mag_sizeaware"] = (
    (df["log_oe"] > 0)
    & (df["residual"] > 0)
    & (df["is_high_conf"])
    & (df["log_oe_pct_in_slice"] >= 0.99)
    & (df["log_oe"] >= log_oe_severity_cut)
    & (df["slice_n"] >= MIN_SLICE_N)
)

def _masked_max(s: pd.Series, mask: pd.Series) -> float:
    x = s[mask]
    return float(x.max()) if len(x) else np.nan

def _masked_p95(s: pd.Series, mask: pd.Series) -> float:
    x = s[mask]
    return float(np.nanpercentile(x, 95)) if len(x) else np.nan

# Provider summary (mag) computed on anomalous rows for severity stats
provider_summary_mag_sizeaware_all = (
    df.groupby(GROUP, dropna=False)
      .apply(lambda g: pd.Series({
          "n_rows": len(g),
          "n_anom_rows_mag": int(g["is_row_anomalous_mag_sizeaware"].sum()),
          "anom_rate_pct_mag": float(g["is_row_anomalous_mag_sizeaware"].mean() * 100),
          "n_unique_codes": int(g["HCPCS_Cd"].nunique()),
          "n_unique_years": int(g["Year"].nunique()),
          "max_log_oe_mag": _masked_max(g["log_oe"], g["is_row_anomalous_mag_sizeaware"]),
          "p95_log_oe_mag": _masked_p95(g["log_oe"], g["is_row_anomalous_mag_sizeaware"]),
          "median_log_oe": float(g["log_oe"].median()),
          "total_services": float(np.nansum(g["services"].to_numpy())),
          "total_benes": float(np.nansum(g["benes"].to_numpy())),
      }), include_groups=False)
      .reset_index()
)

provider_summary_mag_sizeaware_all["provider_anom_score_mag"] = (
    provider_summary_mag_sizeaware_all["n_anom_rows_mag"]
    + 0.25 * provider_summary_mag_sizeaware_all["n_unique_codes"]
    + 0.25 * provider_summary_mag_sizeaware_all["n_unique_years"]
    + 0.10 * provider_summary_mag_sizeaware_all["p95_log_oe_mag"].fillna(0)
)

print("provider_summary_mag_sizeaware_all:", provider_summary_mag_sizeaware_all.shape)

# Optional quick sanity: how many providers have >=1 event in each definition
print("Providers with >=1 robust event:",
      int((provider_summary_robust_sizeaware_all["n_anom_rows_robust"] > 0).sum()))
print("Providers with >=1 shock event:",
      int((provider_summary_mag_sizeaware_all["n_anom_rows_mag"] > 0).sum()))

## Note: Confidence gate differs between anomaly surfacing vs tiering

In the anomaly surfacing notebook, the row-level “event” flags (used to build ANOM.3 provider worklists) used a broader confidence gate:

- **Anomaly surfacing confidence gate**
  - `is_event_conf = (is_high_conf | high_confidence_anomaly_candidate)`

This allowed an event to qualify if it either:
- passed the strict, volume-aware `is_high_conf` rule, **or**
- satisfied the legacy `high_confidence_anomaly_candidate` heuristic (tier-based support + high residual + high OE).

In this tiering notebook, we intentionally simplify the confidence gate to be stricter and more uniform:

- **Tiering confidence gate**
  - `is_event_conf = is_high_conf`

So in tiering, a row must pass:
- support tier in `{high, medium_high}`
- and minimum volume thresholds (`services >= MIN_SERVICES`, `benes >= MIN_BENES`)

### What this changes (and why it is acceptable)

- Tiering will generally produce **fewer** robust/shock “events” than anomaly surfacing, because it no longer admits events that only pass the legacy candidate flag.
- This is intentional for tiering, because tiers should be built on **stable, volume-supported signals** rather than any heuristics tied to extreme residual/OE rules.
- Anomaly surfacing remains the “broad detection” workflow. Tiering is the “stable representation” workflow.

### Important implication

Provider scorecard metrics derived from:
- `is_row_anomalous_robust_sizeaware` and `is_row_anomalous_mag_sizeaware`

are **tiering-specific** and should not be expected to match the anomaly surfacing worklists 1:1.

# TIER.3) Build provider_scorecard_v1 + freeze to artifacts/provider_tiering/

In [ ]:
# ============================================================
# TIER.3) Build provider_scorecard_v1 + freeze to artifacts/provider_tiering/
# Output:
#   - provider_scorecard_v1__{ts}.parquet
#   - provider_scorecard_v1__{ts}.csv
#   - params__{ts}.json + params__{ts}.md + manifest__{ts}.json
# Notes:
#   - Keep ALL providers. Downstream tiering can filter by volume / n_rows.
# ============================================================

from pathlib import Path
import json
import pandas as pd
import numpy as np

# -----------------------------
# 0) Join robust + mag summaries into one scorecard
# -----------------------------
KEY = ["Rndrng_NPI","provider_type","state"]

rob = provider_summary_robust_sizeaware_all.copy()
mag = provider_summary_mag_sizeaware_all.copy()

for c in KEY:
    if c not in rob.columns or c not in mag.columns:
        raise KeyError(f"Missing {KEY} in provider summaries.")

provider_scorecard_v1 = rob.merge(
    mag[KEY + [c for c in mag.columns if c not in KEY]],
    on=KEY,
    how="left",
    suffixes=("_rob","_mag"),
)

# Fill missing mag metrics (providers with no shock events)
for c in ["n_anom_rows_mag","anom_rate_pct_mag","max_log_oe_mag","p95_log_oe_mag","provider_anom_score_mag"]:
    if c in provider_scorecard_v1.columns:
        provider_scorecard_v1[c] = pd.to_numeric(provider_scorecard_v1[c], errors="coerce").fillna(0)

# Helpful derived fields (optional)
provider_scorecard_v1["has_any_robust_event"] = provider_scorecard_v1["n_anom_rows_robust"] > 0
provider_scorecard_v1["has_any_shock_event"] = provider_scorecard_v1["n_anom_rows_mag"] > 0

print("provider_scorecard_v1:", provider_scorecard_v1.shape)
display(provider_scorecard_v1.head(5))

# -----------------------------
# 1) Freeze to disk with params
# -----------------------------
ts = pd.Timestamp.now(tz="America/New_York").strftime("%Y%m%d_%H%M%S")
OUT_DIR = Path("artifacts/provider_tiering") / f"run_{ts}"
OUT_DIR.mkdir(parents=True, exist_ok=True)

PARQUET_PATH = OUT_DIR / f"provider_scorecard_v1__{ts}.parquet"
CSV_PATH     = OUT_DIR / f"provider_scorecard_v1__{ts}.csv"

provider_scorecard_v1.to_parquet(PARQUET_PATH, index=False)
provider_scorecard_v1.to_csv(CSV_PATH, index=False)

PARAMS = {
    "timestamp_et": ts,
    "outputs_dir": str(OUT_DIR),
    "input_eval_path": str(Path("artifacts/eval_universe/eval_scored_DG_V3.parquet")),
    "provider_grain": KEY,
    "tier_feature_layer": {
        "MIN_SERVICES": MIN_SERVICES,
        "MIN_BENES": MIN_BENES,
        "HIGH_CONF_TIERS": sorted(list(HIGH_CONF_TIERS)),
        "slice_cols": ["HCPCS_Cd","Year"],
    },
    "robust_events": {
        "MIN_SLICE_N": MIN_SLICE_N,
        "USE_MIN_EXPECTED_FILTER": USE_MIN_EXPECTED_FILTER,
        "MIN_EXPECTED": MIN_EXPECTED if USE_MIN_EXPECTED_FILTER else None,
        "event_flag": "(log_oe>0) & (residual>0) & (is_high_conf) & (log_oe_pct_in_slice>=0.99) & (slice_n>=MIN_SLICE_N)",
        "provider_score": "n_anom_rows_robust + 0.25*n_unique_codes + 0.25*n_unique_years",
    },
    "shock_events": {
        "MIN_SLICE_N": MIN_SLICE_N,
        "USE_MIN_EXPECTED_FILTER": USE_MIN_EXPECTED_FILTER,
        "MIN_EXPECTED": MIN_EXPECTED if USE_MIN_EXPECTED_FILTER else None,
        "LOG_OE_Q": LOG_OE_Q,
        "log_oe_severity_cut": float(log_oe_severity_cut),
        "event_flag": "(log_oe>0) & (residual>0) & (is_high_conf) & (log_oe_pct_in_slice>=0.99) & (log_oe>=log_oe_severity_cut) & (slice_n>=MIN_SLICE_N)",
        "provider_score": "n_anom_rows_mag + 0.25*n_unique_codes + 0.25*n_unique_years + 0.10*p95_log_oe_mag",
    },
    "files": {
        "parquet": str(PARQUET_PATH),
        "csv": str(CSV_PATH),
    },
}

PARAMS_JSON = OUT_DIR / f"params__{ts}.json"
MANIFEST_JSON = OUT_DIR / f"manifest__{ts}.json"
PARAMS_MD = OUT_DIR / f"params__{ts}.md"

with open(PARAMS_JSON, "w") as f:
    json.dump(PARAMS, f, indent=2)

with open(MANIFEST_JSON, "w") as f:
    json.dump(
        {"timestamp_et": ts, "outputs_dir": str(OUT_DIR),
         "artifacts": [{"name":"provider_scorecard_v1","rows":int(len(provider_scorecard_v1)),"cols":int(provider_scorecard_v1.shape[1]),
                        "parquet": str(PARQUET_PATH), "csv": str(CSV_PATH)}]},
        f, indent=2
    )

md = []
md.append(f"# Provider tiering scorecard (v1) — {ts} ET\n\n")
md.append(f"Output directory: `{OUT_DIR}`\n\n")
md.append("## Artifacts\n")
md.append(f"- Parquet: `{PARQUET_PATH}`\n")
md.append(f"- CSV: `{CSV_PATH}`\n\n")
md.append("## Parameters\n")
md.append("```json\n" + json.dumps(PARAMS, indent=2) + "\n```\n")
PARAMS_MD.write_text("".join(md))

print("✅ Wrote provider_scorecard_v1")
print(" -", PARQUET_PATH)
print(" -", CSV_PATH)
print(" -", PARAMS_JSON)
print(" -", PARAMS_MD)
print(" -", MANIFEST_JSON)

# Sanity checks on TIER.3 output `provider_scorecard_v1`

### SANITY.TIER3.0) we used `suffixes=("_rob","_mag")` in the merge, overlapping column names should be suffixed consistently, and the ones we expect should exist. 

In [ ]:
# SANITY.TIER3.0: quick schema sanity
expected_cols = [
    "Rndrng_NPI","provider_type","state",
    "n_rows_rob","n_anom_rows_robust","anom_rate_pct_robust",
    "n_unique_codes_rob","n_unique_years_rob",
    "n_anom_rows_mag","p95_log_oe_mag","max_log_oe_mag",
    "provider_anom_score_robust","provider_anom_score_mag",
    "has_any_robust_event","has_any_shock_event"
]
missing = [c for c in expected_cols if c not in provider_scorecard_v1.columns]
print("Missing expected cols:", missing)
print("n providers:", len(provider_scorecard_v1))
print("providers w robust events:", int(provider_scorecard_v1["has_any_robust_event"].sum()))
print("providers w shock events:", int(provider_scorecard_v1["has_any_shock_event"].sum()))

### SANITY.TIER3.1) Grain + uniqueness

Confirms the scorecard is truly one row per (Rndrng_NPI, provider_type, state).

Expected: unique key rows == rows, max duplicates per key == 1, n duplicate keys == 0.

In [ ]:
# ============================================================
# SANITY.TIER3.1) Grain + uniqueness
# ============================================================

keys = ["Rndrng_NPI","provider_type","state"]

print("rows:", len(provider_scorecard_v1))
print("unique key rows:", provider_scorecard_v1[keys].drop_duplicates().shape[0])
print("max duplicates per key:", provider_scorecard_v1.groupby(keys).size().max())

dups = provider_scorecard_v1.groupby(keys).size().reset_index(name="n").query("n>1")
print("n duplicate keys:", len(dups))
display(dups.head(10))

Quick sanity check: we had 21,591 unique providers but we have 23,233 unique provider profiles:

In [ ]:
provider_scorecard_v1.Rndrng_NPI.nunique()

- Provider scorecard grain = (Rndrng_NPI, provider_type, state) → 23,366 “provider profiles”
- Unique NPIs = Rndrng_NPI.nunique() → 21,591 providers
- Difference means some NPIs appear under multiple provider_type and/or state (multiple profiles).


##### SANITY.TIER3.1.1) NPIs with multiple provider_type and/or state profiles

In [ ]:
# ============================================================
# SANITY.TIER3.1.1) NPIs with multiple provider_type and/or state profiles
# ============================================================

import pandas as pd

KEY = ["Rndrng_NPI", "provider_type", "state"]

# 1) Confirm counts
print("scorecard rows:", len(provider_scorecard_v1))
print("unique (NPI, provider_type, state):", provider_scorecard_v1[KEY].drop_duplicates().shape[0])
print("unique NPIs:", provider_scorecard_v1["Rndrng_NPI"].nunique())

# 2) NPIs with >1 distinct provider_type and/or state
npi_multi = (
    provider_scorecard_v1
    .groupby("Rndrng_NPI", dropna=False)
    .agg(
        n_profiles=("provider_type", "size"),                 # rows at provider-profile grain
        n_provider_types=("provider_type", pd.Series.nunique),
        n_states=("state", pd.Series.nunique),
        provider_types=("provider_type", lambda s: sorted(set(map(str, s.dropna())))),
        states=("state", lambda s: sorted(set(map(str, s.dropna())))),
    )
    .reset_index()
)

multi_any = npi_multi.query("n_provider_types > 1 or n_states > 1").copy()
print("NPIs with >1 provider_type and/or >1 state:", len(multi_any))

# 3) Split into categories (nice for interpretation)
multi_both = multi_any.query("n_provider_types > 1 and n_states > 1")
multi_pt_only = multi_any.query("n_provider_types > 1 and n_states == 1")
multi_state_only = multi_any.query("n_provider_types == 1 and n_states > 1")

print("  - multi provider_type AND multi state:", len(multi_both))
print("  - multi provider_type only:", len(multi_pt_only))
print("  - multi state only:", len(multi_state_only))

# 4) Show top examples (by number of profiles)
display(
    multi_any.sort_values(["n_profiles", "n_provider_types", "n_states"], ascending=False)
             .head(50)
)

# 5) If you want the exact duplicated profiles for the top few NPIs:
top_npis = multi_any.sort_values("n_profiles", ascending=False).head(10)["Rndrng_NPI"].tolist()
display(
    provider_scorecard_v1.loc[provider_scorecard_v1["Rndrng_NPI"].isin(top_npis), KEY]
    .sort_values(["Rndrng_NPI", "provider_type", "state"])
)

#### SANITY.TIER3.1.2) Do multi-profile NPIs concentrate volume?

In [ ]:
# ============================================================
# SANITY.TIER3.1.2) Do multi-profile NPIs concentrate volume?
# ============================================================

multi_set = set(multi_any["Rndrng_NPI"].astype(str))

tmp = provider_scorecard_v1.copy()
tmp["Rndrng_NPI"] = tmp["Rndrng_NPI"].astype(str)
tmp["is_multi_profile_npi"] = tmp["Rndrng_NPI"].isin(multi_set)

# choose whichever exists in your scorecard
svc_col = "total_services_rob" if "total_services_rob" in tmp.columns else "total_services"
ben_col = "total_benes_rob" if "total_benes_rob" in tmp.columns else "total_benes"

tmp[svc_col] = pd.to_numeric(tmp.get(svc_col, 0), errors="coerce").fillna(0)
tmp[ben_col] = pd.to_numeric(tmp.get(ben_col, 0), errors="coerce").fillna(0)

summary = (
    tmp.groupby("is_multi_profile_npi")
       .agg(
           n_profiles=("Rndrng_NPI", "size"),
           n_unique_npi=("Rndrng_NPI", pd.Series.nunique),
           total_services=(svc_col, "sum"),
           total_benes=(ben_col, "sum"),
       )
       .reset_index()
)

summary["share_services_pct"] = 100 * summary["total_services"] / max(summary["total_services"].sum(), 1e-12)
summary["share_benes_pct"] = 100 * summary["total_benes"] / max(summary["total_benes"].sum(), 1e-12)
display(summary)

### SANITY.TIER3.2) Mag columns: no NaNs, zeros behave correctly

Validates our “left join + fillna(0)” behavior and ensures no accidental NaNs remain in mag columns.

Expected: no NaNs; and among n_anom_rows_mag==0, the severity metrics should be 0 as well.

In [ ]:
# ============================================================
# SANITY.TIER3.2) Mag columns: no NaNs, zeros behave correctly
# ============================================================

mag_cols = ["n_anom_rows_mag","anom_rate_pct_mag","max_log_oe_mag","p95_log_oe_mag","provider_anom_score_mag"]
present = [c for c in mag_cols if c in provider_scorecard_v1.columns]

print("Mag cols present:", present)
print("NaNs in mag cols:")
display(provider_scorecard_v1[present].isna().sum().to_frame("n_na").T)

# Providers with no shock events should have severity stats == 0
no_shock = provider_scorecard_v1["n_anom_rows_mag"] == 0
print("no_shock providers:", int(no_shock.sum()))
print("max_log_oe_mag > 0 among no_shock (should be 0):", int((provider_scorecard_v1.loc[no_shock, "max_log_oe_mag"] > 0).sum()))
print("p95_log_oe_mag > 0 among no_shock (should be 0):", int((provider_scorecard_v1.loc[no_shock, "p95_log_oe_mag"] > 0).sum()))

### SANITY.TIER3.3) Range sanity (hard bounds + “this can’t happen” checks)

Catches dtype issues, sign flips, and nonsense percentages.

Expected: all zeros for the “should be 0” lines.

In [ ]:
# ============================================================
# SANITY.TIER3.3) Range sanity
# ============================================================

df = provider_scorecard_v1

# Counts cannot be negative
for c in ["n_rows_rob","n_anom_rows_robust","n_anom_rows_mag","n_unique_codes_rob","n_unique_years_rob"]:
    if c in df.columns:
        bad = (df[c] < 0).sum()
        print(f"{c}: n_neg =", int(bad))

# Rates should be in [0, 100]
for c in ["anom_rate_pct_robust","anom_rate_pct_mag","pct_high_conf_rows"]:
    if c in df.columns:
        bad_lo = (df[c] < -1e-9).sum()
        bad_hi = (df[c] > 100 + 1e-9).sum()
        print(f"{c}: <0 =", int(bad_lo), "| >100 =", int(bad_hi))

# Logical consistency
print("n_anom_rows_robust > n_rows_rob (should be 0):",
      int((df["n_anom_rows_robust"] > df["n_rows_rob"]).sum()))
print("has_any_robust_event mismatch (should be 0):",
      int(((df["n_anom_rows_robust"] > 0) != df["has_any_robust_event"]).sum()))
print("has_any_shock_event mismatch (should be 0):",
      int(((df["n_anom_rows_mag"] > 0) != df["has_any_shock_event"]).sum()))

### SANITY.TIER3.4) Headline counts match TIER.2

This is the “trust but verify” check.

Expected (our current strict-gate run): 23366, 2691, 150.

In [ ]:
# ============================================================
# SANITY.TIER3.4) Headline counts match TIER.2
# ============================================================

n_prov = len(provider_scorecard_v1)
n_rob  = int((provider_scorecard_v1["n_anom_rows_robust"] > 0).sum())
n_mag  = int((provider_scorecard_v1["n_anom_rows_mag"] > 0).sum())

print("n providers:", n_prov)
print("providers w robust events:", n_rob)
print("providers w shock events:", n_mag)

### SANITY.TIER3.5) Top-of-distribution spot checks

Spot-check “top ends” (does the ranking look plausible?)

This is where we catch “we joined wrong and everything is flat” problems.

What we want to see: a sensible mix of provider types, non-trivial counts/rates, and non-zero shock severities concentrated in the shock list.

In [ ]:
# ============================================================
# SANITY.TIER3.5) Top-of-distribution spot checks
# ============================================================

cols_show = ["Rndrng_NPI","provider_type","state","n_rows_rob","n_anom_rows_robust","anom_rate_pct_robust",
             "n_unique_codes_rob","n_unique_years_rob","provider_anom_score_robust",
             "n_anom_rows_mag","p95_log_oe_mag","max_log_oe_mag","provider_anom_score_mag"]

cols_show = [c for c in cols_show if c in provider_scorecard_v1.columns]

print("\nTop 10 repeat offenders by n_anom_rows_robust:")
display(provider_scorecard_v1.sort_values("n_anom_rows_robust", ascending=False)[cols_show].head(10))

print("\nTop 10 shocks by max_log_oe_mag:")
display(provider_scorecard_v1.sort_values("max_log_oe_mag", ascending=False)[cols_show].head(10))

# TIER.4) Build tier_features_v1 (clean + scaled + defensible)

In [ ]:
# ============================================================
# TIER.4) Build tier_features_v1 (clean + scaled + defensible)
# Output:
#   - tier_features_v1 (all providers, with eligibility flag)
#   - tier_features_v1_clustering (eligible subset, scaled)
# Notes:
#   - Uses provider_scorecard_v1 from TIER.3
#   - Keeps raw columns for reporting, and adds *_w (winsorized) + *_rs (robust-scaled)
# ============================================================

import numpy as np
import pandas as pd

# -----------------------------
# Preconditions
# -----------------------------
if "provider_scorecard_v1" not in globals():
    raise NameError("Missing provider_scorecard_v1. Run TIER.3 first.")

score = provider_scorecard_v1.copy()

KEY = ["Rndrng_NPI", "provider_type", "state"]
for c in KEY:
    if c not in score.columns:
        raise KeyError(f"provider_scorecard_v1 missing key col: {c}")

# -----------------------------
# 0) Select a SMALL feature set for clustering
# -----------------------------
# Cost performance (robust center + upper tail)
# Repeat offender behavior
# Shock severity
# Scale / confidence
FEATURES_RAW = [
    # cost performance
    "median_log_oe_rob",
    "median_residual",
    "p90_log_oe",
    "p90_residual",

    # repeat offender / breadth
    "n_anom_rows_robust",
    "anom_rate_pct_robust",
    "n_unique_codes_rob",
    "n_unique_years_rob",

    # shock severity (already masked to shock events; 0 if none)
    "p95_log_oe_mag",
    "max_log_oe_mag",

    # scale / confidence (use _rob; _mag is identical per our note)
    "total_services_rob",
    "total_benes_rob",
    "pct_high_conf_rows",
]

missing = [c for c in FEATURES_RAW if c not in score.columns]
if missing:
    raise KeyError(f"provider_scorecard_v1 missing required feature cols: {missing}")

# -----------------------------
# 1) Cluster-eligibility gate (defensible)
# -----------------------------
MIN_N_ROWS_FOR_CLUSTER = 50
MIN_SERVICES_FOR_CLUSTER = 200

score["is_cluster_eligible_v1"] = (
    (pd.to_numeric(score["n_rows_rob"], errors="coerce").fillna(0) >= MIN_N_ROWS_FOR_CLUSTER)
    & (pd.to_numeric(score["total_services_rob"], errors="coerce").fillna(0) >= MIN_SERVICES_FOR_CLUSTER)
)

print("Eligibility gate:")
print("  MIN_N_ROWS_FOR_CLUSTER =", MIN_N_ROWS_FOR_CLUSTER)
print("  MIN_SERVICES_FOR_CLUSTER =", MIN_SERVICES_FOR_CLUSTER)
print(score["is_cluster_eligible_v1"].value_counts(dropna=False))

# -----------------------------
# 2) Winsorize heavy-tail features (reduce domination)
# -----------------------------
WINSOR_P_LO = 0.01
WINSOR_P_HI = 0.99

WINSOR_COLS = [
    "median_residual",
    "p90_residual",
    "n_anom_rows_robust",
    "anom_rate_pct_robust",
    "n_unique_codes_rob",
    "total_services_rob",
    "total_benes_rob",
    "p95_log_oe_mag",
    "max_log_oe_mag",
]

def _winsorize(s: pd.Series, p_lo: float, p_hi: float) -> pd.Series:
    x = pd.to_numeric(s, errors="coerce")
    lo = float(x.quantile(p_lo))
    hi = float(x.quantile(p_hi))
    return x.clip(lower=lo, upper=hi)

for c in WINSOR_COLS:
    score[c + "_w"] = _winsorize(score[c], WINSOR_P_LO, WINSOR_P_HI)

# Keep non-winsor columns as-is but numeric
KEEP_ASIS = [c for c in FEATURES_RAW if c not in WINSOR_COLS]
for c in KEEP_ASIS:
    score[c + "_w"] = pd.to_numeric(score[c], errors="coerce")

# -----------------------------
# 3) Robust scaling (median/IQR) for clustering
# -----------------------------
SCALE_COLS_W = [c + "_w" for c in FEATURES_RAW]

def _robust_scale(s: pd.Series) -> pd.Series:
    x = pd.to_numeric(s, errors="coerce")
    med = float(x.median())
    q1 = float(x.quantile(0.25))
    q3 = float(x.quantile(0.75))
    iqr = (q3 - q1)
    if iqr == 0 or not np.isfinite(iqr):
        std = float(x.std())
        denom = std if std and np.isfinite(std) else 1.0
    else:
        denom = iqr
    return (x - med) / denom

for c in SCALE_COLS_W:
    score[c.replace("_w", "_rs")] = _robust_scale(score[c])

SCALE_COLS_RS = [c.replace("_w", "_rs") for c in SCALE_COLS_W]

# -----------------------------
# 4) Assemble tier_features_v1 tables
# -----------------------------
tier_features_v1 = score[KEY + ["is_cluster_eligible_v1"] + FEATURES_RAW + SCALE_COLS_W + SCALE_COLS_RS].copy()

tier_features_v1_clustering = tier_features_v1.loc[
    tier_features_v1["is_cluster_eligible_v1"]
].copy()

before = len(tier_features_v1_clustering)
tier_features_v1_clustering = tier_features_v1_clustering.dropna(subset=SCALE_COLS_RS)
after = len(tier_features_v1_clustering)

print("\nBuilt tier_features_v1:", tier_features_v1.shape)
print("Eligible for clustering:", int(tier_features_v1["is_cluster_eligible_v1"].sum()))
print("Dropped for NA in scaled cols:", before - after)

print("\nScaled column quick sanity (median should be ~0):")
for c in ["median_log_oe_rob_rs", "n_anom_rows_robust_rs", "total_services_rob_rs", "max_log_oe_mag_rs"]:
    if c in tier_features_v1_clustering.columns:
        print(f"  {c}: median={float(tier_features_v1_clustering[c].median()):.3f} "
              f"p05={float(tier_features_v1_clustering[c].quantile(0.05)):.3f} "
              f"p95={float(tier_features_v1_clustering[c].quantile(0.95)):.3f}")

### Quick sanity check: confirm `_rob` and `_mag` versions match exactly

In [ ]:
# optional: confirm _rob and _mag versions match exactly (if present)
if "total_services_mag" in score.columns:
    print("total_services_rob == total_services_mag:",
          bool((pd.to_numeric(score["total_services_rob"])
               == pd.to_numeric(score["total_services_mag"])).all()))
if "total_benes_mag" in score.columns:
    print("total_benes_rob == total_benes_mag:",
          bool((pd.to_numeric(score["total_benes_rob"])
               == pd.to_numeric(score["total_benes_mag"])).all()))

#### Is `total_services_rob_rs` behaving weirdly because the scaler is broken, or is it behaving exactly as expected because the eligibility gate changes the population?

In [ ]:
(score.
 groupby("is_cluster_eligible_v1")["total_services_rob_rs"].
 agg("median"))

`total_services_rob_rs` is near 0 for the full provider population the scaler was fit on, but strongly positive for cluster-eligible providers because the eligibility gate intentionally selects higher-volume providers. This confirms the scaling is behaving as expected and highlights that volume would dominate clustering distances if included as a feature.

# TIER.4.a) Value-focused clustering inputs (Option A)

In [ ]:
# ============================================================
# TIER.4.a) Value-focused clustering inputs (Option A)
# Goal:
#   - Use a SMALL, defensible set of *_rs columns
#   - Exclude volume features from distance (total_services_rob, total_benes_rob)
# Output:
#   - tier_features_v1_clustering_A (eligible subset, scaled cols only)
#   - CLUSTER_COLS_A (list of columns fed into clustering)
# Notes:
#   - Keeps the SAME eligibility gate from TIER.4 (volume used only for eligibility)
# ============================================================

import pandas as pd

# -----------------------------
# Preconditions
# -----------------------------
if "tier_features_v1_clustering" not in globals():
    raise NameError("Missing tier_features_v1_clustering. Run TIER.4 first.")

dfc = tier_features_v1_clustering.copy()

# -----------------------------
# 0) Choose value-focused clustering columns (scaled)
# -----------------------------
# Cost performance (log scale preferred)
# Repeat-offender behavior (frequency + rate)
# Shock severity (pick ONE to avoid redundancy)
# Confidence (optional, but often helps)
CLUSTER_COLS_A = [
    "median_log_oe_rob_rs",
    "p90_log_oe_rs",
    "n_anom_rows_robust_rs",
    "anom_rate_pct_robust_rs",
    "p95_log_oe_mag_rs",      # shock severity (choose this OR max_log_oe_mag_rs)
    "pct_high_conf_rows_rs",
]

# CLUSTER_COLS_A[4] = "max_log_oe_mag_rs"
# Optional: if we prefer max shock over p95 shock, swap:

missing = [c for c in CLUSTER_COLS_A if c not in dfc.columns]
if missing:
    raise KeyError(f"Missing required *_rs cols for Option A clustering: {missing}")

# -----------------------------
# 1) Build the clustering matrix (keep keys for traceability)
# -----------------------------
KEY = ["Rndrng_NPI", "provider_type", "state"]
tier_features_v1_clustering_A = dfc[KEY + CLUSTER_COLS_A].copy()

# Final NA guard (should be zero if TIER.4 already dropped NA in scaled cols)
before = len(tier_features_v1_clustering_A)
tier_features_v1_clustering_A = tier_features_v1_clustering_A.dropna(subset=CLUSTER_COLS_A)
after = len(tier_features_v1_clustering_A)

print("Option A clustering matrix:", tier_features_v1_clustering_A.shape)
print("Dropped rows due to NA in CLUSTER_COLS_A:", before - after)

# Quick spread read (what will drive distance inside eligible set)
print("\nOption A spread check (eligible only):")
for c in CLUSTER_COLS_A:
    print(f"  {c}: p05={float(tier_features_v1_clustering_A[c].quantile(0.05)):.3f} "
          f"median={float(tier_features_v1_clustering_A[c].median()):.3f} "
          f"p95={float(tier_features_v1_clustering_A[c].quantile(0.95)):.3f}")

# TIER.4.JUSTIFY) Why we exclude some features from clustering (TIER.4 -> TIER.4.a)

In [ ]:
# ============================================================
# TIER.4.JUSTIFY) Why we exclude some features from clustering (TIER.4 -> TIER.4.a)
# What this does:
#   A) Shows redundancy (high correlations) among candidate features
#   B) Shows which features dominate Euclidean distance (pairwise distance share)
#   C) Shows how strongly excluded features track "scale" (services/benes), i.e. size clusters
# Outputs:
#   - tables we can screenshot + a few headline prints
# Notes:
#   - Uses the robust-scaled columns produced in TIER.4 (the *_rs columns)
#   - Uses a random sample of providers (keeps runtime low)
# ============================================================

import numpy as np
import pandas as pd

# -----------------------------
# Preconditions
# -----------------------------
if "tier_features_v1_clustering" not in globals():
    raise NameError("Missing tier_features_v1_clustering. Run TIER.4 first.")

dfc = tier_features_v1_clustering.copy()

# These are the scaled columns that correspond to our original FEATURES_RAW set in TIER.4
FULL_RS = [c + "_rs" for c in [
    "median_log_oe_rob",
    "median_residual",
    "p90_log_oe",
    "p90_residual",
    "n_anom_rows_robust",
    "anom_rate_pct_robust",
    "n_unique_codes_rob",
    "n_unique_years_rob",
    "p95_log_oe_mag",
    "max_log_oe_mag",
    "total_services_rob",
    "total_benes_rob",
    "pct_high_conf_rows",
]]

# Our Option A columns (value-focused)
CLUSTER_COLS_A = [
    "median_log_oe_rob_rs",
    "p90_log_oe_rs",
    "n_anom_rows_robust_rs",
    "anom_rate_pct_robust_rs",
    "p95_log_oe_mag_rs",      # (choose this OR max_log_oe_mag_rs)
    "pct_high_conf_rows_rs",
]

# Excluded = full minus option A
EXCLUDED_RS = [c for c in FULL_RS if c not in CLUSTER_COLS_A]

missing_full = [c for c in FULL_RS if c not in dfc.columns]
missing_a = [c for c in CLUSTER_COLS_A if c not in dfc.columns]
if missing_full:
    raise KeyError(f"Missing FULL_RS cols in tier_features_v1_clustering: {missing_full}")
if missing_a:
    raise KeyError(f"Missing CLUSTER_COLS_A cols in tier_features_v1_clustering: {missing_a}")

# -----------------------------
# Helper: clean names for printing
# -----------------------------
def _base_name(col_rs: str) -> str:
    return col_rs[:-3] if col_rs.endswith("_rs") else col_rs

# -----------------------------
# A) Redundancy check: correlations (eligible set only)
# -----------------------------
corr = dfc[FULL_RS].corr(numeric_only=True)
# Flag "near duplicates" using abs corr threshold
CORR_THR = 0.85
pairs = []
cols = list(corr.columns)
for i in range(len(cols)):
    for j in range(i + 1, len(cols)):
        r = corr.iloc[i, j]
        if np.isfinite(r) and abs(r) >= CORR_THR:
            pairs.append((_base_name(cols[i]), _base_name(cols[j]), float(r)))
corr_pairs = pd.DataFrame(pairs, columns=["feat_1", "feat_2", "corr"]).sort_values("corr", key=lambda s: s.abs(), ascending=False)

print("A) Redundancy check (abs corr >= %.2f) among robust-scaled features:" % CORR_THR)
display(corr_pairs.head(30))

# -----------------------------
# B) Distance domination check: who drives Euclidean distance?
#    Approach:
#      - sample N providers
#      - compute pairwise squared diffs per feature
#      - compute each feature's share of total squared distance across all pairs
# -----------------------------
RNG = np.random.default_rng(7)
MAX_N = 2500  # tune if needed; 1500-3000 works well
n = min(MAX_N, len(dfc))
idx = RNG.choice(dfc.index.to_numpy(), size=n, replace=False)
X = dfc.loc[idx, FULL_RS].to_numpy(dtype="float64")

# ensure finite (should be, but guard)
mask_finite = np.isfinite(X).all(axis=1)
X = X[mask_finite]
n_eff = X.shape[0]

# Pairwise distance share computation without building full N x N matrices:
# We'll compute sum over upper-triangle pairs of (x_i - x_j)^2 per feature using:
# sum_{i<j} (xi - xj)^2 = n*sum(x^2) - (sum x)^2, applied feature-wise.
# This gives total squared-distance mass contributed by each feature.
sum_x = np.nansum(X, axis=0)
sum_x2 = np.nansum(X**2, axis=0)
feat_mass = n_eff * sum_x2 - (sum_x**2)  # vector length = n_features
total_mass = float(np.nansum(feat_mass))
share = feat_mass / total_mass if total_mass > 0 else np.zeros_like(feat_mass)

dist_share = pd.DataFrame({
    "feature": [_base_name(c) for c in FULL_RS],
    "distance_share": share.astype(float),
}).sort_values("distance_share", ascending=False)

print("\nB) Distance domination check (eligible set, sample n=%d):" % n_eff)
print("   Interpretation: higher share => more influence on Euclidean distance.")
display(dist_share)

# Convenience view: group shares by INCLUDED vs EXCLUDED (Option A)
incl_share = float(dist_share.loc[dist_share["feature"].isin([_base_name(c) for c in CLUSTER_COLS_A]), "distance_share"].sum())
excl_share = float(dist_share.loc[dist_share["feature"].isin([_base_name(c) for c in EXCLUDED_RS]), "distance_share"].sum())
print("\nDistance-share totals:")
print(f"  Included in Option A: {incl_share:.3f}")
print(f"  Excluded in Option A: {excl_share:.3f}")

# -----------------------------
# C) "Size clustering" diagnostic:
#    If excluded features track volume strongly, they will steer clusters toward size.
#    We quantify how much each feature correlates with total_services_rob_rs.
# -----------------------------
SIZE_COL = "total_services_rob_rs"
if SIZE_COL in dfc.columns:
    size_corr = []
    s = dfc[SIZE_COL]
    for c in FULL_RS:
        r = float(pd.Series(dfc[c]).corr(s))
        size_corr.append((_base_name(c), r))
    size_corr_df = pd.DataFrame(size_corr, columns=["feature", "corr_with_total_services_rs"])
    size_corr_df["abs_corr"] = size_corr_df["corr_with_total_services_rs"].abs()
    size_corr_df = size_corr_df.sort_values("abs_corr", ascending=False)

    print("\nC) Size clustering diagnostic: correlation with total_services_rob_rs (eligible set)")
    display(size_corr_df.drop(columns=["abs_corr"]).head(20))

    # Highlight excluded features that are strongly tied to size
    SIZE_THR = 0.50
    strong_size = size_corr_df[(size_corr_df["abs_corr"] >= SIZE_THR)].copy()
    print(f"\nFeatures with |corr| >= {SIZE_THR:.2f} vs total_services_rob_rs (these tend to create 'size clusters'):")
    display(strong_size[["feature", "corr_with_total_services_rs"]])

# -----------------------------
# D) Quick “why excluded” checklist output (for our Markdown narrative)
# -----------------------------
reason_rows = []
top_dom = dist_share.head(6)["feature"].tolist()

for feat_rs in FULL_RS:
    feat = _base_name(feat_rs)
    included = feat_rs in CLUSTER_COLS_A
    dom = feat in top_dom

    # redundancy proxy: does it appear in the high-corr pair list?
    redundant = bool(((corr_pairs["feat_1"] == feat) | (corr_pairs["feat_2"] == feat)).any())

    # size-tied proxy (if available)
    size_tied = None
    if SIZE_COL in dfc.columns:
        r = float(size_corr_df.loc[size_corr_df["feature"] == feat, "corr_with_total_services_rs"].iloc[0])
        size_tied = abs(r) >= 0.50

    reason_rows.append({
        "feature": feat,
        "in_option_A": included,
        "distance_dominant_top6": dom,
        "redundant_abs_corr_ge_%.2f" % CORR_THR: redundant,
        "size_tied_abs_corr_ge_0.50": bool(size_tied) if size_tied is not None else None,
    })

reasons = pd.DataFrame(reason_rows).sort_values(["in_option_A", "distance_dominant_top6"], ascending=[False, False])
print("\nD) Feature decision helper (use this to write our Markdown):")
display(reasons)

# TIER.4.b) Value-focused clustering inputs (Option A, LOCKED)

In [ ]:
# ============================================================
# TIER.4.b) Value-focused clustering inputs (Option A, LOCKED)
# Goal:
#   - Freeze the final clustering columns AFTER justification (TIER.4.JUSTIFY)
#   - Keep the SAME eligibility gate from TIER.4 (volume used only for eligibility)
# Output:
#   - tier_features_v1_clustering_LOCKED (eligible subset, keys + locked scaled cols)
#   - CLUSTER_COLS_LOCKED (final list used by clustering)
# Notes:
#   - This is the "source of truth" for clustering inputs going forward.
#   - TIER.4.a remains the audit trail of the initial proposal.
# ============================================================

import pandas as pd

# -----------------------------
# Preconditions
# -----------------------------
if "tier_features_v1_clustering" not in globals():
    raise NameError("Missing tier_features_v1_clustering. Run TIER.4 first.")

dfc = tier_features_v1_clustering.copy()

KEY = ["Rndrng_NPI", "provider_type", "state"]
for c in KEY:
    if c not in dfc.columns:
        raise KeyError(f"tier_features_v1_clustering missing key col: {c}")

# -----------------------------
# 0) LOCK the final clustering columns
# -----------------------------
# Justification basis (from TIER.4.JUSTIFY):
# - Exclude size drivers: total_services_rob_rs dominates distance; total_benes_rob_rs strongly size-tied.
# - Exclude size proxies: n_unique_codes_rob_rs and total_benes_rob_rs are highly correlated and size-tied.
# - Prefer robust tail: p95_log_oe_mag_rs over max_log_oe_mag_rs to avoid single-event domination.
CLUSTER_COLS_LOCKED = [
    "median_log_oe_rob_rs",     # typical performance vs expected (log scale)
    "p90_log_oe_rs",            # upper-tail tendency (log scale)
    "n_anom_rows_robust_rs",    # repeat-offender frequency
    "anom_rate_pct_robust_rs",  # repeat-offender rate (normalizes by footprint)
    "p95_log_oe_mag_rs",        # shock severity (robust)
    "pct_high_conf_rows_rs",    # confidence texture (not pure size)
]

# Optional: if we later want multi-year stability in tiers, prefer adding:
# "n_unique_years_rob_rs"
# (kept out by default to keep the locked set minimal)

missing = [c for c in CLUSTER_COLS_LOCKED if c not in dfc.columns]
if missing:
    raise KeyError(f"Missing required *_rs cols for LOCKED clustering: {missing}")

# -----------------------------
# 1) Explicit guardrails to prevent accidental "size clustering"
# -----------------------------
# These should NOT appear in the locked feature set.
FORBIDDEN_SIZE_COLS = {
    "total_services_rob_rs",
    "total_benes_rob_rs",
    "n_unique_codes_rob_rs",
    "median_residual_rs",
    "p90_residual_rs",
    "max_log_oe_mag_rs",
}

overlap = set(CLUSTER_COLS_LOCKED) & FORBIDDEN_SIZE_COLS
assert len(overlap) == 0, f"LOCKED set includes forbidden size/instability cols: {sorted(list(overlap))}"

# -----------------------------
# 2) Build the final clustering matrix (keys + locked cols)
# -----------------------------
tier_features_v1_clustering_LOCKED = dfc[KEY + CLUSTER_COLS_LOCKED].copy()

before = len(tier_features_v1_clustering_LOCKED)
tier_features_v1_clustering_LOCKED = tier_features_v1_clustering_LOCKED.dropna(subset=CLUSTER_COLS_LOCKED)
after = len(tier_features_v1_clustering_LOCKED)

print("LOCKED clustering matrix:", tier_features_v1_clustering_LOCKED.shape)
print("Dropped rows due to NA in CLUSTER_COLS_LOCKED:", before - after)

# -----------------------------
# 3) Quick distribution readout (what will drive distance, eligible only)
# -----------------------------
print("\nLOCKED spread check (eligible only):")
for c in CLUSTER_COLS_LOCKED:
    print(
        f"  {c}: p05={float(tier_features_v1_clustering_LOCKED[c].quantile(0.05)):.3f} "
        f"median={float(tier_features_v1_clustering_LOCKED[c].median()):.3f} "
        f"p95={float(tier_features_v1_clustering_LOCKED[c].quantile(0.95)):.3f}"
    )

# For convenience downstream
print("\nFinal clustering columns (LOCKED):")
for c in CLUSTER_COLS_LOCKED:
    print(" -", c)

# Clustering workflow summary

## TIER.4 → TIER.4.JUSTIFY → TIER.4.a → TIER.4.b (how clustering inputs were built and finalized)

### TIER.4) Build `tier_features_v1` (clean + scaled + defensible)
**Goal:** Create one table that is both:
- **reportable** (keeps raw provider metrics for interpretation), and
- **cluster-ready** (adds winsorized + robust-scaled versions of the same metrics).

**What we did:**
- Started from `provider_scorecard_v1` (one row per provider at grain `(Rndrng_NPI, provider_type, state)`).
- Defined a **cluster-eligibility gate** (used only to decide who is eligible for clustering, not to define “value”):
  - `n_rows_rob >= MIN_N_ROWS_FOR_CLUSTER`
  - `total_services_rob >= MIN_SERVICES_FOR_CLUSTER`
- Selected an intentionally small set of **raw features** (`FEATURES_RAW`) spanning:
  - cost performance (center + tail),
  - repeat offender behavior,
  - shock severity,
  - scale + confidence.
- Winsorized heavy-tailed raw columns at `[1%, 99%]` to prevent extreme outliers from dominating.
- Robust-scaled every feature using **median / IQR** to produce `*_rs` columns suitable for Euclidean distance.
- Produced:
  - `tier_features_v1` (all providers, raw + winsor + scaled + eligibility flag),
  - `tier_features_v1_clustering` (eligible subset, scaled columns present, no NaNs).

**Key point:** In TIER.4, **volume features were included**, so clustering at that stage can unintentionally become “size-based”.

---

### TIER.4.JUSTIFY) Diagnostics to justify excluding some features from clustering distance
**Goal:** Provide quantitative evidence for which features should or should not be allowed to drive clustering distance.

**What we tested:**
1. **Redundancy check:** high absolute correlations among scaled features (e.g., “these carry the same information”).
2. **Distance domination check:** estimated each feature’s share of squared Euclidean distance on the eligible set.
   - This directly answers: “Which columns actually decide nearest neighbors?”
3. **Size-tie check:** correlation of each feature with `total_services_rob_rs`.
   - This answers: “Will this feature push the model to form volume clusters?”

**What we found (headline):**
- `total_services_rob` dominated distance almost entirely (about **97.6%** distance share), meaning clustering would mostly separate providers by volume.
- `total_benes_rob` and `n_unique_codes_rob` were strongly size-tied and/or redundant, reinforcing “size clustering”.
- Shock severity features were sparse under the stricter confidence gate, so they contribute little for most providers, which is fine as long as we treat “shockiness” as an optional differentiator.

**Conclusion:** If the tiering story is “value-focused”, volume belongs in the **eligibility gate**, not in the **distance metric**.

---

### TIER.4.a) Option A candidate: value-focused clustering inputs
**Goal:** Propose a minimal, defensible set of scaled (`*_rs`) features that reflect “value and behavior”, not “size”.

**What we did:**
- Kept the same eligible set from TIER.4 (volume gate still applies).
- Proposed a candidate clustering column list (`CLUSTER_COLS_A`) using scaled features only, excluding:
  - `total_services_rob_rs`, `total_benes_rob_rs`, `n_unique_codes_rob_rs`, and other “size” drivers.

**Why this step exists:** It is the “proposal layer” that stays readable and auditable before we lock the final version.

---

### TIER.4.b) LOCKED clustering inputs (final, justified)
**Goal:** Freeze the exact clustering feature set we will use going forward, explicitly derived from the justification diagnostics.

**What we locked:**
- Final clustering columns (`CLUSTER_COLS_LOCKED`) fed into clustering:
  - `median_log_oe_rob_rs` (typical cost performance vs expected)
  - `p90_log_oe_rs` (upper-tail tendency)
  - `n_anom_rows_robust_rs` (repeat offender frequency)
  - `anom_rate_pct_robust_rs` (repeat offender rate, normalizes by footprint)
  - `p95_log_oe_mag_rs` (shock severity, robust)
  - `pct_high_conf_rows_rs` (confidence texture)

**What we explicitly excluded from distance (by guardrail):**
- Volume / size drivers: `total_services_rob_rs`, `total_benes_rob_rs`
- Size proxy / redundancy: `n_unique_codes_rob_rs`
- Instability / single-event domination: `max_log_oe_mag_rs`
- Residual-level magnitudes that can behave like scale: `p90_residual_rs`, `median_residual_rs`

**Outputs:**
- `tier_features_v1_clustering_LOCKED` = keys + locked scaled columns for eligible providers only.
- This is the single source of truth for any clustering (TIER.5+).

**Interpretation note from the spread check:**
- Several “event count” style features have median ~0 in the eligible set, meaning many eligible providers still have zero robust/shock events. That is expected. The clustering still works because variation lives in the upper tail (p95) and the other continuous performance features.

# TIER.5) Cluster eligible providers + attach cluster labels + freeze artifacts

In [ ]:
# ============================================================
# TIER.5) Cluster eligible providers + attach cluster labels + freeze artifacts
# Uses:
#   - tier_features_v1_clustering_LOCKED (eligible subset, keys + scaled cols)
#   - CLUSTER_COLS_LOCKED (final clustering columns)
# Produces:
#   - tier_clusters_v1 (eligible providers with cluster_id + distance-to-centroid)
#   - provider_scorecard_v1 (augmented with cluster_id_v1, cluster_label_v1 placeholders)
#   - tier_features_v1 (augmented with cluster_id_v1)
# Writes:
#   - tier_clusters_v1__{ts}.parquet/csv
#   - provider_scorecard_v1_labeled__{ts}.parquet/csv
#   - params__{ts}.json/md + manifest__{ts}.json
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import json

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# -----------------------------
# Preconditions
# -----------------------------
need_globals = [
    "tier_features_v1",
    "provider_scorecard_v1",
    "tier_features_v1_clustering_LOCKED",
    "CLUSTER_COLS_LOCKED",
]
for g in need_globals:
    if g not in globals():
        raise NameError(f"Missing {g}. Run earlier tier steps first.")

KEY = ["Rndrng_NPI", "provider_type", "state"]
for c in KEY:
    if c not in tier_features_v1.columns:
        raise KeyError(f"tier_features_v1 missing key col: {c}")
    if c not in tier_features_v1_clustering_LOCKED.columns:
        raise KeyError(f"tier_features_v1_clustering_LOCKED missing key col: {c}")
    if c not in provider_scorecard_v1.columns:
        raise KeyError(f"provider_scorecard_v1 missing key col: {c}")

missing_cols = [c for c in CLUSTER_COLS_LOCKED if c not in tier_features_v1_clustering_LOCKED.columns]
if missing_cols:
    raise KeyError(f"Locked clustering frame missing: {missing_cols}")

dfc = tier_features_v1_clustering_LOCKED.copy()

# X matrix (eligible only)
X = dfc[CLUSTER_COLS_LOCKED].to_numpy(dtype="float64")
if not np.isfinite(X).all():
    raise ValueError("Non-finite values found in clustering matrix. Check TIER.4 scaling/NA drops.")

print("Eligible rows for clustering:", len(dfc))
print("Clustering columns (LOCKED):", CLUSTER_COLS_LOCKED)

# -----------------------------
# 0) Choose K (small sweep)
# -----------------------------
K_MIN = 2
K_MAX = 10
RANDOM_STATE = 7
N_INIT = 20

rows = []
for k in range(K_MIN, K_MAX + 1):
    km = KMeans(n_clusters=k, n_init=N_INIT, random_state=RANDOM_STATE)
    labels = km.fit_predict(X)
    inertia = float(km.inertia_)
    sil = float(silhouette_score(X, labels)) if k >= 2 else np.nan
    rows.append({"k": k, "inertia": inertia, "silhouette": sil})

k_sweep = pd.DataFrame(rows)
display(k_sweep)

# Simple default: pick best silhouette
K_CHOSEN = int(k_sweep.sort_values("silhouette", ascending=False).iloc[0]["k"])
print("K_CHOSEN (by max silhouette):", K_CHOSEN)

# Optional quick plots (kept simple, default colors)
fig = plt.figure(figsize=(10, 4))
ax = fig.add_subplot(111)
ax.plot(k_sweep["k"], k_sweep["inertia"], marker="o")
ax.set_title("K sweep: inertia (lower is better)")
ax.set_xlabel("k")
ax.set_ylabel("inertia")
plt.tight_layout()
plt.show()

fig = plt.figure(figsize=(10, 4))
ax = fig.add_subplot(111)
ax.plot(k_sweep["k"], k_sweep["silhouette"], marker="o")
ax.set_title("K sweep: silhouette (higher is better)")
ax.set_xlabel("k")
ax.set_ylabel("silhouette")
plt.tight_layout()
plt.show()

# -----------------------------
# 1) Fit final KMeans + compute distance to centroid
# -----------------------------
km_final = KMeans(n_clusters=K_CHOSEN, n_init=N_INIT, random_state=RANDOM_STATE)
cluster_id = km_final.fit_predict(X)

# distance to assigned centroid (Euclidean in scaled space)
centroids = km_final.cluster_centers_
dists = np.linalg.norm(X - centroids[cluster_id], axis=1)

tier_clusters_v1 = dfc[KEY].copy()
tier_clusters_v1["cluster_id_v1"] = cluster_id.astype(int)
tier_clusters_v1["dist_to_centroid_v1"] = dists.astype(float)

print("\nCluster sizes (eligible only):")
display(tier_clusters_v1["cluster_id_v1"].value_counts().sort_index().rename("n"))

print("\nDistance-to-centroid (smaller = more typical within cluster):")
display(tier_clusters_v1["dist_to_centroid_v1"].describe(percentiles=[0.5, 0.9, 0.95, 0.99]))

# -----------------------------
# 2) Attach cluster_id back to tier_features_v1 (all providers)
# -----------------------------
tier_features_v1 = tier_features_v1.copy()
tier_features_v1["cluster_id_v1"] = np.nan

tier_features_v1 = tier_features_v1.merge(
    tier_clusters_v1[KEY + ["cluster_id_v1", "dist_to_centroid_v1"]],
    on=KEY,
    how="left",
)

# -----------------------------
# 3) Attach cluster_id back to provider_scorecard_v1 (all providers)
# -----------------------------
provider_scorecard_v1 = provider_scorecard_v1.copy()
provider_scorecard_v1 = provider_scorecard_v1.merge(
    tier_clusters_v1[KEY + ["cluster_id_v1", "dist_to_centroid_v1"]],
    on=KEY,
    how="left",
)

# Placeholder labels (we will map clusters to tiers in TIER.6)
provider_scorecard_v1["cluster_label_v1"] = provider_scorecard_v1["cluster_id_v1"].map(lambda x: f"cluster_{int(x)}" if pd.notna(x) else "ineligible")

print("\nAll providers: cluster coverage")
print(provider_scorecard_v1["cluster_id_v1"].isna().value_counts(dropna=False).rename(index={True:"no_cluster (ineligible)", False:"clustered"}))

# -----------------------------
# 4) Freeze artifacts
# -----------------------------
ts = pd.Timestamp.now(tz="America/New_York").strftime("%Y%m%d_%H%M%S")
OUT_DIR = Path("artifacts/provider_tiering") / f"run_{ts}"
OUT_DIR.mkdir(parents=True, exist_ok=True)

def _write(df: pd.DataFrame, stem: str) -> dict:
    pq = OUT_DIR / f"{stem}__{ts}.parquet"
    csv = OUT_DIR / f"{stem}__{ts}.csv"
    df.to_parquet(pq, index=False)
    df.to_csv(csv, index=False)
    return {"name": stem, "rows": int(len(df)), "cols": int(df.shape[1]), "parquet": str(pq), "csv": str(csv)}

manifest = []
manifest.append(_write(tier_clusters_v1, "tier_clusters_v1"))
manifest.append(_write(provider_scorecard_v1, "provider_scorecard_v1_labeled"))
manifest.append(_write(k_sweep, "k_sweep_v1"))

PARAMS = {
    "timestamp_et": ts,
    "outputs_dir": str(OUT_DIR),
    "eligibility_gate": {
        "source": "TIER.4",
        "note": "clustered only eligible providers; ineligible keep NaN cluster_id",
    },
    "clustering": {
        "algorithm": "KMeans",
        "random_state": RANDOM_STATE,
        "n_init": N_INIT,
        "k_sweep": {"k_min": K_MIN, "k_max": K_MAX},
        "k_chosen": K_CHOSEN,
        "cluster_cols_locked": CLUSTER_COLS_LOCKED,
        "distance": "Euclidean in robust-scaled feature space",
    },
    "files": manifest,
}

PARAMS_JSON = OUT_DIR / f"params__{ts}.json"
MANIFEST_JSON = OUT_DIR / f"manifest__{ts}.json"
PARAMS_MD = OUT_DIR / f"params__{ts}.md"

with open(PARAMS_JSON, "w") as f:
    json.dump(PARAMS, f, indent=2)

with open(MANIFEST_JSON, "w") as f:
    json.dump({"timestamp_et": ts, "outputs_dir": str(OUT_DIR), "artifacts": manifest}, f, indent=2)

md = []
md.append(f"# Provider tiering clustering (v1) — {ts} ET\n\n")
md.append(f"Output directory: `{OUT_DIR}`\n\n")
md.append("## Artifacts\n")
for item in manifest:
    md.append(f"- {item['name']}: {item['rows']} × {item['cols']}\n")
    md.append(f"  - Parquet: `{item['parquet']}`\n")
    md.append(f"  - CSV: `{item['csv']}`\n")
md.append("\n## Parameters\n")
md.append("```json\n" + json.dumps(PARAMS, indent=2) + "\n```\n")
PARAMS_MD.write_text("".join(md))

print("\n✅ Wrote TIER.5 clustering artifacts")
print(" -", OUT_DIR)
print(" -", PARAMS_JSON)
print(" -", PARAMS_MD)
print(" -", MANIFEST_JSON)
display(pd.DataFrame(manifest))

# Interpreting TIER.5 End-to-End

## What TIER.5 is trying to do

We already built a `provider_scorecard_v1` (all providers) and a `tier_features_v1_clustering_LOCKED` matrix (eligible providers only, scaled “value” features only).

TIER.5 answers:
* Do the eligible providers naturally group into a small number of behavior patterns based on “value signals” (cost performance + repeat anomalies + shock severity + confidence)?
* If yes, how many groups (k) is a reasonable choice?
* Once we pick k, assign each eligible provider a cluster label and attach it back to the full provider scorecard for product use and reporting.
* Freeze these outputs so tiering is reproducible.

That’s exactly what our TIER.5 outputs show.

---

## First, confirm what we actually clustered

### Eligible rows for clustering: 7,488
This matches our TIER.4 eligibility gate result:
* **Eligible** = `is_cluster_eligible_v1 == True`
* **Ineligible** = everyone else (15,878)

So the clustering is not “all providers in the universe”. It is “only providers with enough evidence to be compared fairly”.

### Clustering columns (LOCKED)
We clustered on:
* `median_log_oe_rob_rs`
* `p90_log_oe_rs`
* `n_anom_rows_robust_rs`
* `anom_rate_pct_robust_rs`
* `p95_log_oe_mag_rs`
* `pct_high_conf_rows_rs`

All are:
* winsorized (where needed),
* robust-scaled,
* and intentionally exclude volume as a distance driver (volume only gates eligibility, not clustering distance).

So the clusters are meant to reflect “behavior/value style”, not “big vs small practice”.

---

## Interpreting the K-sweep table

We computed two diagnostic metrics across k = 2..10:

### 1) Inertia (lower is better)
Inertia is the sum of squared distances from each point to its assigned cluster centroid. As k increases, inertia always decreases (we can always fit data better with more clusters).

**Our inertia curve:**
* k=2: 12,911
* k=3: 10,227
* k=4: 7,556
* k=5: 6,610
* k=6: 5,779
* k=10: 3,999

**How to read it:**
* The big drops happen early (2→3→4). After k≈4–5, improvements taper.
* This is the classic “elbow” idea: where adding more clusters stops buying us much.
* Our curve suggests an elbow around **k=4 or k=5**.

### 2) Silhouette (higher is better)
Silhouette measures how well-separated clusters are:
* **near 1.0** = very clean separation
* **~0** = overlapping clusters
* **negative** = lots of misassignment

**Our silhouette:**
* k=2: 0.600
* k=3: 0.556
* k=4: 0.529
* k=5: 0.526
* k=6: 0.350
* k=7: 0.347
* k=8: 0.346
* k=9: 0.336
* k=10: 0.352

**How to read it:**
* k=2 is best by a lot.
* k=3–5 are still “good” (0.52–0.56).
* k≥6 collapses to ~0.35, meaning clusters start to overlap and become less coherent.

This is a strong signal that the data naturally separates into **2 big groupings**, and forcing more than 5 starts producing unstable or overlapping partitions.

---

## Why the code chose K_CHOSEN = 2

Our code chose:
**K_CHOSEN (by max silhouette): 2**

That is a defensible “default” rule when we want the cleanest separation.

**Interpretation:**
* The eligible provider population splits into **two distinct behavioral regimes** under our value-focused features.
* Higher k values might be useful later for product nuance, but they reduce separation quality.

**A practical way to frame it:**
* **k=2** is our clean binary segmentation (two “tiers” or two “archetypes”).
* **k=4 or k=5** could be our “subtiering” if we later want more granularity, but we would be trading clarity for nuance.

---

## Interpreting the cluster sizes

**Cluster sizes among eligible providers:**
* cluster 0: 5,662 (about 75.6%)
* cluster 1: 1,826 (about 24.4%)

**This means:**
* Most eligible providers fall into one broad “typical” pattern.
* A smaller, meaningful minority forms a distinctly different pattern.

This is usually exactly what we want from tiering. A smaller “special” group that differs systematically.

At this point, the next question is: what does cluster 1 represent? Higher anomaly rate? Worse cost performance? Higher shocks? More consistent? That is TIER.6 territory (cluster profiling).

---

## Interpreting dist_to_centroid_v1

We computed distance from each provider to its assigned cluster centroid:

**Summary:**
* median ~0.874
* 95th percentile ~2.36
* 99th percentile ~3.70
* max ~16.46

**How to read it:**
* **Small distance** means the provider is very “typical” for its cluster.
* **Large distance** means the provider is unusual even within its own cluster.

**This is extremely useful downstream:**
* We can use `dist_to_centroid_v1` as a **cluster confidence score**.
* For UI, we might show tier label plus a “typical vs unusual” indicator.
* The max being 16.46 tells us a few providers are extreme outliers even after robust scaling, which is expected.

---

## Coverage across all providers

**“All providers: cluster coverage”:**
* clustered: 7,488
* no_cluster (ineligible): 15,878

This is exactly what we want. We are explicitly saying:
* “We only tier providers when we have enough rows and services to be fair.”
* “Everyone else stays un-tiered until more evidence exists.”

That’s a very defensible product posture.

---

## What the plots are telling us

### Inertia plot
Shows diminishing returns as k increases. Supports elbow around 4–5, but does not “choose” k by itself.

### Silhouette plot
Strongly favors k=2. Also shows k=3–5 are reasonable, but after 5 quality drops sharply.

This combination is actually ideal:
* silhouette says “2 big groups are real”
* inertia says “if we want subtleties, 4–5 is where diminishing returns begin”

---

## What we froze, and why it matters

Our manifest indicates we saved:

1. **`tier_clusters_v1` (7,488 × 5)**
   This is our eligible subset with:
   * keys
   * cluster label
   * distance to centroid
   * maybe chosen k metadata depending on our schema

2. **`provider_scorecard_v1_labeled` (23,366 × 34)**
   This is the full scorecard with added fields like:
   * `cluster_id_v1` for eligible providers
   * `no_cluster` (ineligible) for the rest
   * `dist_to_centroid_v1` possibly null for ineligible

3. **`k_sweep_v1` (9 × 3)**
   The audit artifact: k, inertia, silhouette.

Plus params and manifests. This makes the tiering step reproducible and auditable, which is exactly what we want for the engine.

---

## Key takeaways from our current results

* **Our “value-focused” feature design worked.** We got high silhouettes (0.60 at k=2, still >0.52 at k=4–5), which indicates meaningful structure.
* **There are two very clean provider archetypes** among cluster-eligible providers.
* **We have a natural path to more granular tiering** if we want it later:
  * k=2 = simplest, clearest story
  * k=4–5 = subtiers (still decent silhouette)
* **The eligibility gate is doing its job:** tiering is applied only where we have enough evidence.

# TIER.6) Cluster profiling (v1) + (optional) tier ordering + freeze artifacts

In [ ]:
# ============================================================
# TIER.6) Cluster profiling (v1) + (optional) tier ordering + freeze artifacts
# Goal:
#   - Produce an interpretable "cluster profile" table (medians + p95) on RAW (unscaled) features
#   - Attach business-friendly cluster definitions (cluster_definition_v1)
#   - (Optional later) Map clusters -> Tier_A/Tier_B for UI ordering. Not written in this cell.
#   - Freeze: cluster_profiles_v1_compact + provider_scorecard_v1_clustered
# Inputs:
#   - provider_scorecard_v1 (must already include cluster_id_v1 + dist_to_centroid_v1 + cluster_label_v1 from TIER.5)
#   - CLUSTER_COLS_LOCKED (scaled feature names used in clustering)
# Notes:
#   - Profiles include: the 6 LOCKED features (raw versions) + a few reporting-only scale cols.
#   - We still compute a data-driven ordering score for clusters (tier_order_score_v1) for reference,
#     but we do NOT create or write tier_label_v1 in this cell.
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path
import json

# -----------------------------
# Preconditions
# -----------------------------
if "provider_scorecard_v1" not in globals():
    raise NameError("Missing provider_scorecard_v1. Run TIER.3 first.")
if "CLUSTER_COLS_LOCKED" not in globals():
    raise NameError("Missing CLUSTER_COLS_LOCKED. Run TIER.4.b first.")
if "cluster_id_v1" not in provider_scorecard_v1.columns:
    raise NameError("provider_scorecard_v1 missing cluster_id_v1. Run TIER.5 first.")
if "cluster_label_v1" not in provider_scorecard_v1.columns:
    raise NameError("provider_scorecard_v1 missing cluster_label_v1. Run TIER.5 first.")

score = provider_scorecard_v1.copy()

# -----------------------------
# NEW (minimal): add cluster_definition_v1 (business-friendly labels)
# -----------------------------
CLUSTER_DEFINITION_MAP_V1 = {
    "cluster_0": "Typical cost behavior (no robust tail events)",
    "cluster_1": "Elevated anomaly burden (robust tail events present)",
    # keep anything else explicit
    "ineligible": "Ineligible (insufficient evidence for clustering)",
}
score["cluster_definition_v1"] = score["cluster_label_v1"].astype(str).map(CLUSTER_DEFINITION_MAP_V1).fillna("Unmapped cluster")

# Keep eligible clusters only for cluster-to-cluster comparisons
eligible = score[score["cluster_label_v1"].astype(str).str.startswith("cluster_")].copy()
if len(eligible) == 0:
    raise ValueError("No clustered rows found. Check TIER.5 outputs.")

# -----------------------------
# 0) Define which raw columns we profile
# -----------------------------
# Convert locked *_rs column names -> raw feature names (as stored in provider_scorecard_v1)
LOCKED_RAW = [c.replace("_rs", "") for c in CLUSTER_COLS_LOCKED]

# Safety: ensure the raw features exist
missing_raw = [c for c in LOCKED_RAW if c not in score.columns]
if missing_raw:
    raise KeyError(f"provider_scorecard_v1 missing raw columns needed for profiling: {missing_raw}")

# Reporting-only columns (not used for clustering distance)
REPORT_COLS = []
for c in [
    "total_services_rob",
    "total_benes_rob",
    "n_rows_rob",
    "provider_anom_score_robust",
    "n_unique_codes_rob",
    "n_unique_years_rob",
    "n_anom_rows_mag",
    "provider_anom_score_mag",
]:
    if c in score.columns and c not in REPORT_COLS:
        REPORT_COLS.append(c)

PROFILE_COLS = LOCKED_RAW + REPORT_COLS

# -----------------------------
# 1) Helper stats
# -----------------------------
def _p05(x: pd.Series) -> float:
    return float(x.quantile(0.05))

def _p95(x: pd.Series) -> float:
    return float(x.quantile(0.95))

AGGS = ["count", "mean", "median", _p05, _p95]

# -----------------------------
# 2) Core cluster profile table (eligible clusters only)
# -----------------------------
cluster_profiles_v1 = (
    eligible
    .groupby(["cluster_label_v1", "cluster_definition_v1"], dropna=False)[PROFILE_COLS]
    .agg(AGGS)
)

# Make it easier to read: sort clusters by median "repeat-offender severity"
# (primary: median n_anom_rows_robust, then median p90_log_oe)
sort_keys = []
if ("n_anom_rows_robust", "median") in cluster_profiles_v1.columns:
    sort_keys.append(("n_anom_rows_robust", "median"))
if ("p90_log_oe", "median") in cluster_profiles_v1.columns:
    sort_keys.append(("p90_log_oe", "median"))

if sort_keys:
    # sort on the aggregated columns, preserving MultiIndex index (cluster_label, definition)
    order = (
        cluster_profiles_v1[sort_keys]
        .sort_values(sort_keys, ascending=[False] * len(sort_keys))
        .index.tolist()
    )
    cluster_profiles_v1 = cluster_profiles_v1.loc[order]

print("Cluster profile table (eligible clusters only):")
with pd.option_context("display.max_columns", None):
    display(cluster_profiles_v1)

# -----------------------------
# 3) Compact executive view (medians + p95 only)
# -----------------------------
cols_keep = []
for col in PROFILE_COLS:
    for stat in ["median", "_p95"]:
        if (col, stat) in cluster_profiles_v1.columns:
            cols_keep.append((col, stat))

cluster_profiles_v1_compact = cluster_profiles_v1.loc[:, cols_keep].copy()

# Flatten columns for readability
cluster_profiles_v1_compact.columns = [f"{c}__{s}" for (c, s) in cluster_profiles_v1_compact.columns]

print("\nCluster profile (compact): medians + p95")
with pd.option_context("display.max_columns", None):
    display(cluster_profiles_v1_compact)

# -----------------------------
# 4) Data-driven ordering score (reference only; no tier_label output here)
# -----------------------------
# Choose raw columns for ordering (must exist)
ORDER_COLS = []
for c in ["median_log_oe_rob", "p90_log_oe", "n_anom_rows_robust", "anom_rate_pct_robust"]:
    if c in LOCKED_RAW:
        ORDER_COLS.append(c)

if not ORDER_COLS:
    raise ValueError("No ORDER_COLS available for ordering. Check CLUSTER_COLS_LOCKED / raw mapping.")

cluster_medians = (
    eligible.groupby("cluster_label_v1")[ORDER_COLS]
    .median()
    .copy()
)

z = (cluster_medians - cluster_medians.mean()) / (cluster_medians.std(ddof=0).replace(0, 1.0))
cluster_medians["tier_order_score_v1"] = z.sum(axis=1)

cluster_rank = cluster_medians.sort_values("tier_order_score_v1", ascending=True).index.tolist()

print("\nCluster ordering (best-to-worst by tier_order_score_v1, reference only):")
display(pd.DataFrame({
    "cluster_label_v1": cluster_rank,
    "cluster_definition_v1": [CLUSTER_DEFINITION_MAP_V1.get(c, "Unmapped cluster") for c in cluster_rank],
    "tier_order_score_v1": [float(cluster_medians.loc[c, "tier_order_score_v1"]) for c in cluster_rank],
}))

# -----------------------------
# 4.1) Output scorecard with cluster fields only (no tier labels)
# -----------------------------
provider_scorecard_v1_clustered = score.copy()

print("\nCluster label coverage (all providers):")
display(provider_scorecard_v1_clustered["cluster_label_v1"].value_counts(dropna=False).rename("n"))

# -----------------------------
# 5) Freeze artifacts (cluster profiles + clustered scorecard)
# -----------------------------
ts = pd.Timestamp.now(tz="America/New_York").strftime("%Y%m%d_%H%M%S")
OUT_DIR = Path("artifacts/provider_tiering") / f"run_{ts}"
OUT_DIR.mkdir(parents=True, exist_ok=True)

def _write(df: pd.DataFrame, stem: str) -> dict:
    pq = OUT_DIR / f"{stem}__{ts}.parquet"
    csv = OUT_DIR / f"{stem}__{ts}.csv"
    df.to_parquet(pq, index=False)
    df.to_csv(csv, index=False)
    return {"name": stem, "rows": int(len(df)), "cols": int(df.shape[1]), "parquet": str(pq), "csv": str(csv)}

manifest = []
# Save compact profile for easy consumption
manifest.append(_write(cluster_profiles_v1_compact.reset_index(), "cluster_profiles_v1_compact"))
manifest.append(_write(provider_scorecard_v1_clustered, "provider_scorecard_v1_clustered"))

PARAMS = {
    "timestamp_et": ts,
    "outputs_dir": str(OUT_DIR),
    "profiling": {
        "profile_cols": PROFILE_COLS,
        "locked_scaled_cols": CLUSTER_COLS_LOCKED,
        "locked_raw_cols": LOCKED_RAW,
        "aggs": ["count", "mean", "median", "p05", "p95"],
        "groupby": ["cluster_label_v1", "cluster_definition_v1"],
    },
    "cluster_definition_map_v1": CLUSTER_DEFINITION_MAP_V1,
    "ordering_reference_only": {
        "order_cols": ORDER_COLS,
        "tier_order_score_v1": "sum of z-scored cluster medians across ORDER_COLS (lower = better)",
        "cluster_rank_best_to_worst": cluster_rank,
    },
    "files": manifest,
}

PARAMS_JSON = OUT_DIR / f"params__{ts}.json"
MANIFEST_JSON = OUT_DIR / f"manifest__{ts}.json"
PARAMS_MD = OUT_DIR / f"params__{ts}.md"

with open(PARAMS_JSON, "w") as f:
    json.dump(PARAMS, f, indent=2)

with open(MANIFEST_JSON, "w") as f:
    json.dump({"timestamp_et": ts, "outputs_dir": str(OUT_DIR), "artifacts": manifest}, f, indent=2)

md = []
md.append(f"# Provider tiering: cluster profiling (v1) — {ts} ET\n\n")
md.append(f"Output directory: `{OUT_DIR}`\n\n")
md.append("## Artifacts\n")
for item in manifest:
    md.append(f"- {item['name']}: {item['rows']} × {item['cols']}\n")
    md.append(f"  - Parquet: `{item['parquet']}`\n")
    md.append(f"  - CSV: `{item['csv']}`\n")
md.append("\n## Cluster definitions\n")
md.append("```json\n" + json.dumps(CLUSTER_DEFINITION_MAP_V1, indent=2) + "\n```\n")
md.append("\n## Parameters\n")
md.append("```json\n" + json.dumps(PARAMS, indent=2) + "\n```\n")
PARAMS_MD.write_text("".join(md))

print("\n✅ Wrote TIER.6 profiling artifacts (cluster-only)")
print(" -", OUT_DIR)
print(" -", PARAMS_JSON)
print(" -", PARAMS_MD)
print(" -", MANIFEST_JSON)
display(pd.DataFrame(manifest))

# Interpreting the Cluster Profile Tables (TIER.6, cluster-only)

## 1) What these tables are

We built two related tables from **eligible providers only** (the 7,488 providers that passed the TIER.4 eligibility gate and therefore received a cluster assignment):

**A) Full cluster profile table (wide MultiIndex stats)**
* Grouped by: `cluster_label_v1` and `cluster_definition_v1`
* For each group, computed: `count`, `mean`, `median`, `p05`, `p95` for a set of columns.

**B) Compact profile table (executive view)**
* Same groupings, but only shows `median` and `p95` (because these are stable and easy to communicate).
* This is our “cluster card” table.

### What columns are being summarized

The 6 “LOCKED” clustering features (raw versions), i.e., the original, human-interpretable metrics behind the scaled columns used by KMeans:
* **Cost behavior:** `median_log_oe_rob`, `p90_log_oe`
* **Repeat-offender burden:** `n_anom_rows_robust`, `anom_rate_pct_robust`
* **Shock propensity:** `p95_log_oe_mag`
* **Confidence coverage:** `pct_high_conf_rows`

**Reporting-only context columns** (not used in distance, but crucial for interpretation):
* **Scale:** `total_services_rob`, `total_benes_rob`, `n_rows_rob`
* **Breadth:** `n_unique_codes_rob`, `n_unique_years_rob`
* **Our internal scores:** `provider_anom_score_robust`, `provider_anom_score_mag`
* **Shock counts:** `n_anom_rows_mag`

So these tables answer:
> “What does each cluster look like, in raw business terms?”

---

## 2) The big picture result: KMeans found a natural “0 events vs some events” split

We have two eligible clusters:

**`cluster_0`: Typical cost behavior (no robust tail events)**
* `n_anom_rows_robust` median = 0, p95 = 0
* `anom_rate_pct_robust` median = 0, p95 = 0

**`cluster_1`: Elevated anomaly burden (robust tail events present)**
* `n_anom_rows_robust` median = 1, p95 = 3
* `anom_rate_pct_robust` median ≈ 1.01%, p95 ≈ 3.36%

This is the single clearest finding in the entire table.

### Interpretation
The clustering space is dominated by whether a provider has **any robust extreme events at all** under our strict, auditable definition. Because “0 vs >0” is such a strong boundary in the data, `k=2` is not just a statistical artifact. It is a meaningful segmentation.

### What it means operationally
* `cluster_0` is our “eligible but clean” bucket.
* `cluster_1` is our “eligible and flagged” bucket.

This lines up perfectly with our chosen labels.

---

## 3) Cost behavior: both clusters are near neutral on the median, but `cluster_1` has a heavier upper tail

Look at the two cost-behavior metrics:

**Median behavior (`median_log_oe_rob`)**
* `cluster_0` median: 0.000363
* `cluster_1` median: 0.000293
These are both essentially ~0. **Translation:** typical (median) provider behavior is close to expected in both clusters.

**Upper-tail behavior (`p90_log_oe`)**
* `cluster_0` median: 0.0886 (p95 0.2127)
* `cluster_1` median: 0.1059 (p95 0.2490)
So `cluster_1` has a modestly higher “worst 10%” profile.

### Interpretation
The segmentation is not “all bad vs all good” on central tendency. It is “who actually produces tail events” plus a slight upward shift in the upper tail among those providers.

This is a strong, defensible story: we are not punishing a provider for being slightly above expected on average. We are flagging them for **tail behavior** and **repeat tail behavior**.

---

## 4) Shock propensity: mostly absent overall, but concentrated in the flagged cluster’s tail

**Shock severity proxy (`p95_log_oe_mag`)**
* `cluster_0`: median 0, p95 0
* `cluster_1`: median 0, p95 0.7789

Because `p95_log_oe_mag` is computed on **shock-event rows only** (and is 0 for providers with none), “median 0” is expected. Most providers do not have shock events even if they have robust events. But the non-zero p95 in `cluster_1` tells us something important:

### Interpretation
Shock events exist, but they are **rare**. When shocks appear, they mostly show up **inside `cluster_1`** (the anomaly-burden bucket). That supports our narrative that `cluster_1` is not just “repeat offenders.” It is also the bucket where we’d look first for shock-type outliers.

---

## 5) Confidence and scale: `cluster_1` providers are bigger and have higher high-confidence coverage

Even though we excluded volume from distance in Option A, scale still shows up in profiling because it is correlated with opportunity and stability.

**Confidence share (`pct_high_conf_rows`)**
* `cluster_0` median: 48.39%
* `cluster_1` median: 54.57%

**Scale (medians)**
* `total_services_rob`: `cluster_0` 16,414 vs `cluster_1` 151,954
* `total_benes_rob`: `cluster_0` 3,764 vs `cluster_1` 6,734
* `n_rows_rob`: `cluster_0` 81 vs `cluster_1` 128

### Interpretation
`cluster_1` is disproportionately made of **large, high-confidence providers**. That does not “cause” the clustering (because volume is not a clustering feature in Option A), but it helps explain why these providers separate cleanly:
* they have more stable estimation (more high-conf rows),
* more opportunities to generate robust tail events,
* and enough data to be eligible in the first place.

This is a nuance we can state plainly:
> “We cluster only providers with enough evidence. Among those, the providers that actually generate robust tail events tend to be larger and more frequently high-confidence.”

---

## Cluster ordering table: what it is and why it exists

**“Cluster ordering (best-to-worst by `tier_order_score_v1`, reference only)”**

This table is **not a new model** and it does **not change the clusters**. It is a small helper that answers:
> “If I want to sort clusters from ‘more typical’ to ‘more concerning’, what order is consistent with the cluster profiles?”

### What is `tier_order_score_v1`?
We compute it like this:
1. Take each cluster’s **median** values for a small set of “value severity” columns:
   * `median_log_oe_rob`
   * `p90_log_oe`
   * `n_anom_rows_robust`
   * `anom_rate_pct_robust`
2. Standardize those cluster-level medians across clusters (z-score). This avoids one metric dominating just because it has a larger numeric scale.
3. Sum the standardized medians.

So:
* **More negative** `tier_order_score_v1` = “better” (lower tail burden and lower over-expected metrics).
* **More positive** `tier_order_score_v1` = “worse” (more repeat-offender burden and higher upper tail).

With 2 clusters, this is basically a consistent ordering rule:
* `cluster_0` gets -2.0, `cluster_1` gets +2.0.

### Why is it “reference only”?
Because:
* The clusters are already defined by KMeans.
* The ordering score is only a deterministic summary that tells us how to **present** clusters in a UI or in a report.

It is a presentational convenience, not an analytic dependency.

---

## Why we avoid Gold/Silver labels at this stage

The core reason is **scope honesty**.

Our current clusters are derived from:
* model-based cost signals (log(O/E), residual-derived features),
* tail-event logic,
* repeat offender counts,
* plus a strict evidence gate.

They are **not** derived from:
* outcomes, quality measures, readmissions, complications, adherence,
* patient risk, clinical appropriateness,
* or an externally validated “value” ground truth.

So calling them Gold/Silver would imply a quality/value judgment we have not modeled.

### What we *can* claim now, safely and accurately
We can say:
* **`cluster_0`:** “Typical cost behavior under this benchmark. No robust tail events in high-confidence rows.”
* **`cluster_1`:** “Elevated anomaly burden. Robust tail events appear repeatedly in high-confidence rows.”

These labels match what the clustering actually did. They are operational and defensible.

### If we later want “Gold/Silver”
We can earn that naming only after we add one of these:
* a separate “value score” that blends cost and appropriate quality proxies (if available), or
* an explicit policy that says “Gold means low anomaly burden **and** adequate volume **and** stable performance over time,” etc.

For now, cluster-based labels are the correct choice.

---

## Coverage: what’s eligible vs not eligible

Our cluster coverage table:
* **`cluster_0`:** 5,662
* **`cluster_1`:** 1,826
* **`ineligible`:** 15,878

**Interpretation:**
Most providers are `ineligible` by design because we enforce “sufficient evidence” for tiering. That is a feature, not a bug. It prevents meaningless segmentation on tiny denominators.

---

## The one-line executive takeaway

Our tiering v1 is:
> A defensible segmentation of evidence-eligible providers into “typical cost behavior” vs “elevated anomaly burden,” with shock-like behavior appearing primarily inside the elevated-burden bucket.

That is strong and product-relevant, without overclaiming quality.

# TIER.6.A) One-slide "cluster label card" table

In [ ]:
# ============================================================
# TIER.6.A) One-slide "cluster label card" table (cluster-only)
# For each cluster (eligible only):
#   - n providers
#   - median n_anom_rows_robust
#   - median anom_rate_pct_robust
#   - p95 p90_log_oe
#   - share with any shock event (% with n_anom_rows_mag > 0)
#   - median total_services_rob
#
# Minimal update vs prior:
#   - use provider_scorecard_v1 (cluster-only pipeline)
#   - prefer existing cluster_definition_v1 if present (fallback to map)
# ============================================================

import numpy as np
import pandas as pd

if "provider_scorecard_v1_clustered" in globals():
    score = provider_scorecard_v1_clustered.copy()
elif "provider_scorecard_v1" in globals():
    score = provider_scorecard_v1.copy()
else:
    raise NameError("Missing provider_scorecard_v1 (or provider_scorecard_v1_clustered). Run TIER.3+ first.")

need = [
    "cluster_label_v1",
    "n_anom_rows_robust",
    "anom_rate_pct_robust",
    "p90_log_oe",
    "n_anom_rows_mag",
    "total_services_rob",
]
missing = [c for c in need if c not in score.columns]
if missing:
    raise KeyError(f"Missing required cols for cluster card: {missing}")

# Eligible clusters only
eligible = score[score["cluster_label_v1"].astype(str).str.startswith("cluster_")].copy()
if len(eligible) == 0:
    raise ValueError("No eligible clustered rows found. Run TIER.5 to assign clusters.")

# Fallback labels (used only if cluster_definition_v1 is missing)
CLUSTER_LABEL_MAP = {
    "cluster_0": "Typical cost behavior (no robust tail events)",
    "cluster_1": "Elevated anomaly burden (robust tail events present)",
}

def _p95(x: pd.Series) -> float:
    return float(x.quantile(0.95))

cluster_card = (
    eligible
    .assign(has_any_shock_event=pd.to_numeric(eligible["n_anom_rows_mag"], errors="coerce").fillna(0) > 0)
    .groupby("cluster_label_v1", dropna=False)
    .agg(
        n_providers=("cluster_label_v1", "size"),
        median_n_anom_rows_robust=("n_anom_rows_robust", "median"),
        median_anom_rate_pct_robust=("anom_rate_pct_robust", "median"),
        median_p90_log_oe=("p90_log_oe", "median"),
        pct_with_any_shock_event=("has_any_shock_event", lambda s: float(s.mean() * 100)),
        median_total_services_rob=("total_services_rob", "median"),
    )
    .reset_index()
)

# Prefer existing definition column if it exists, otherwise map from labels
if "cluster_definition_v1" in score.columns:
    defn = (
        eligible[["cluster_label_v1", "cluster_definition_v1"]]
        .drop_duplicates(subset=["cluster_label_v1"])
    )
    cluster_card = cluster_card.merge(defn, on="cluster_label_v1", how="left")
else:
    cluster_card["cluster_definition_v1"] = cluster_card["cluster_label_v1"].map(CLUSTER_LABEL_MAP).fillna(cluster_card["cluster_label_v1"])

# Nice ordering: typical first, burden second (if present)
order = ["cluster_0", "cluster_1"]
cluster_card["__order"] = cluster_card["cluster_label_v1"].apply(lambda x: order.index(x) if x in order else 999)
cluster_card = cluster_card.sort_values("__order").drop(columns="__order")

# Light formatting for executive readability
cluster_card["pct_with_any_shock_event"] = cluster_card["pct_with_any_shock_event"].round(2)
cluster_card["median_anom_rate_pct_robust"] = cluster_card["median_anom_rate_pct_robust"].round(3)
cluster_card["median_p90_log_oe"] = cluster_card["median_p90_log_oe"].round(3)
cluster_card["median_total_services_rob"] = cluster_card["median_total_services_rob"].round(1)

with pd.option_context("display.max_columns", None):
    display(cluster_card)

## What this table is

This is a **one-slide definition card** for the **eligible providers only** (the 7,488 providers that passed our clustering eligibility gate). It summarizes, per cluster:

* how many providers are in the cluster,
* how often they trigger **robust tail events** (repeat-offender behavior),
* how intense their upper-tail over-expected behavior looks (`p90_log_oe`),
* whether “shock” events show up at all,
* and how large they are (services volume, reporting-only).

So this table is meant to make our cluster definitions immediately defensible.

---

## Interpret each column

### `cluster_label_v1` + `cluster_definition_v1`

We already gave the right names:
* `cluster_0` = Typical cost behavior (no robust tail events)
* `cluster_1` = Elevated anomaly burden (robust tail events present)

These are not quality labels. They are **behavioral labels** about cost deviation patterns under our strict rules.

### `n_providers`

* `cluster_0`: 5,662 providers
* `cluster_1`: 1,826 providers

Among eligible providers, that’s roughly:
* `cluster_0` ≈ 75.6%
* `cluster_1` ≈ 24.4%

So about 1 in 4 eligible providers land in the “repeat-offender present” bucket.

### `median_n_anom_rows_robust`

This is the cleanest separator.

* **`cluster_0` median = 0.0**
  The typical provider in `cluster_0` has **zero** robust tail events.
* **`cluster_1` median = 1.0**
  The typical provider in `cluster_1` has **at least one** robust tail event.

A “robust tail event” here is not just “high cost”. It means a row satisfied our strict event definition (directionality, high-confidence gate, within-slice top 1% `log_oe`, `slice_n` validity, and denominator hygiene if applied upstream).

This is why our segmentation is so interpretable. The cluster split is basically “0 events” vs “non-zero events” in the eligible population.

### `median_anom_rate_pct_robust`

This normalizes frequency by the provider’s row count (`n_rows_rob`), so it answers:

> “What fraction of this provider’s rows are robust tail events?”

* **`cluster_0` median = 0.000%**
  Consistent with median count = 0.
* **`cluster_1` median = 1.013%**
  This is very coherent with our row-level definition, because “extreme” is built around **top 1% within slice**. So a 1-ish percent rate is exactly what we expect to see for a provider that periodically lands in that tail.

**Mini-example:**
* If a provider has ~100 rows, 1% is about 1 row flagged.
* That matches `cluster_1`’s median `n_anom_rows_robust` = 1.

So `cluster_1` is not “tons of flags”. It is “a small but meaningful tail footprint that repeats.”

### `median_p90_log_oe`

This is the median of the provider-level statistic `p90_log_oe` inside each cluster.

Interpretation of `p90_log_oe`:
*	For each provider, we take the 90th percentile of `log_oe` across that provider’s rows.
*	That gives a “provider upper-tail” summary: what the provider’s higher-cost tail looks like (roughly their worst 10% of rows).
*	Then, within each cluster, we take the median across providers of that value.

So it answers:

“For a typical provider in this cluster, how elevated is their upper-tail over-expected behavior?”

Values:
*	`cluster_0`: `median_p90_log_oe` = 0.089
*	`cluster_1`: `median_p90_log_oe` = 0.106

These are close, but `cluster_1` is modestly higher.

To translate `log_oe` into an approximate O/E multiplier, use exp(log_oe):
*	exp(0.089) ≈ 1.09x
*	exp(0.106) ≈ 1.11x

So: the typical provider in `cluster_1` has a slightly heavier upper tail, consistent with this being the elevated-anomaly-burden cluster.

Important nuance:
The difference is small because the clustering is not primarily separating providers by “who has the biggest over-expected tail.” The primary separator is the presence and frequency of robust tail events (`n_anom_rows_robust`, `anom_rate_pct_robust`). `median_p90_log_oe` is supportive context that helps show `cluster_1` is not only “more flagged,” but also slightly more elevated in its upper-tail cost behavior.

### `pct_with_any_shock_event`

This is:
% of providers in the cluster with `n_anom_rows_mag > 0`

* `cluster_0`: 0.00%
* `cluster_1`: 7.34%

This is a big interpretability win:
* `cluster_0` is not just “no robust events”. It is also basically “no shocks at all”.
* shocks exist almost entirely inside the repeat-offender cluster, but still only for a minority of it.

**Back-of-envelope:**
* 7.34% of 1,826 ≈ **134 providers** with at least one shock event (roughly consistent with our earlier shock counts being in the low hundreds under the stricter gate).

So we can say:

> “Shock-prone providers are a subset of the elevated-anomaly-burden group, not a separate cluster in v1.”

That is exactly what our TIER.6 profiles suggested.

### `median_total_services_rob`

This is reporting-only, but it matters for interpretation.

* `cluster_0` median services = 16,414.5
* `cluster_1` median services = 151,953.5

That is a massive size difference.

**What it means (carefully):**
* Even though we **excluded volume from distance** in Option A, volume still correlates with things like breadth, opportunity to appear in extreme slices, and stability of estimates.
* Uur eligibility gate already ensures minimum evidence, but within eligible, `cluster_1` skews much larger.

So our honest narrative is:

> “Tiering v1 is a segmentation of eligible providers, and the elevated-anomaly cluster is also disproportionately large-volume providers. That is not a flaw. It reflects that large providers have both more measurement stability and more opportunity for repeated tail events. We control this by using strict event definitions and by using volume only for eligibility, not for distance.”

---

## The clean takeaways we can put on a slide

* **`cluster_0` (Typical):** Most eligible providers show **no robust tail events** under strict gating, and **no shock events**.
* **`cluster_1` (Elevated anomaly burden):** A substantial minority of eligible providers show **repeat robust tail events** (median 1, median rate ~1%), and this cluster contains **essentially all shock providers** (7.34% have ≥1 shock event).
* The segmentation is **behavioral and defensible**, not a “quality grade”. It is about **cost deviation patterns** given our learned expected-cost benchmark and strict confidence rules.

# TIER.6.B1) Distribution overlap (eligible only): ECDF of `n_anom_rows_robust` by cluster

In [ ]:
# ============================================================
# TIER.6.B1) Distribution overlap (eligible only): ECDF of n_anom_rows_robust by cluster
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

if "provider_scorecard_v1" not in globals():
    raise NameError("Missing provider_scorecard_v1. Run TIER.3+ first.")

score = provider_scorecard_v1.copy()
eligible = score[score["cluster_label_v1"].astype(str).str.startswith("cluster_")].copy()

if len(eligible) == 0:
    raise ValueError("No eligible clustered rows found. Run TIER.5 to assign clusters.")

fig = plt.figure(figsize=(10, 6))
ax = fig.add_subplot(111)

for cl, g in eligible.groupby("cluster_label_v1", dropna=False):
    x = pd.to_numeric(g["n_anom_rows_robust"], errors="coerce").fillna(0).to_numpy()
    x = np.sort(x)
    y = np.arange(1, len(x) + 1) / len(x)
    ax.plot(x, y, label=cl)

ax.set_title("Eligible providers: ECDF of robust extreme-event counts by cluster")
ax.set_xlabel("n_anom_rows_robust")
ax.set_ylabel("Cumulative share of providers (ECDF)")
ax.legend(frameon=False, loc="best")
plt.tight_layout()
plt.show()

### 1) What an ECDF means in this plot

For any x-value on the horizontal axis (`n_anom_rows_robust`), the y-value is:
* the share of providers in that cluster whose `n_anom_rows_robust` is **≤ x**

So if a curve is at 0.90 at x=2, it means 90% of providers in that cluster have 2 or fewer robust anomalous rows.

Because `n_anom_rows_robust` is an integer count, the ECDF naturally looks “steppy”.

---

### 2) Interpreting `cluster_0` (blue)

The blue curve is essentially a vertical jump at x=0 up to y=1.0.

That means:
* ~100% of `cluster_0` providers have `n_anom_rows_robust = 0`.
* There is essentially no mass at 1, 2, 3, etc.

So `cluster_0` is not “low counts”. It is **“no robust extreme events at all”**.

This is why our label “Typical cost behavior (no robust tail events)” is perfectly aligned with the data.

---

### 3) Interpreting `cluster_1` (orange)

The orange curve starts at x=1 and climbs in steps until it reaches 1.0 around x≈10.

Key reads (approximate from the curve shape):
* **At x=1**, the ECDF jumps to roughly **~0.68–0.70**
  * *Meaning:* about 70% of `cluster_1` providers have 1 or fewer robust events. Since `cluster_1` was defined as the non-zero group, this mostly means “exactly 1”.
* **At x=2**, the ECDF is roughly **~0.88–0.90**
  * *Meaning:* about 90% have 2 or fewer events.
* **At x=3**, the ECDF is roughly **~0.95–0.96**
  * *Meaning:* only ~4–5% have more than 3 events.
* The tail continues out to ~10, but that’s a very small fraction of providers.

So `cluster_1` is not “tons of anomalies”. It is “a provider has at least one robust tail event, and usually only 1–2, with a small heavy tail up to ~10”.

That is a very defensible “monitoring” bucket.

---

### 4) Why this plot is such strong evidence for our segmentation

This ECDF shows **near-zero overlap** between clusters at x=0:
* `cluster_0` is entirely at 0
* `cluster_1` starts at 1 (meaning essentially none of `cluster_1` is at 0)

So the clustering didn’t do something fuzzy or ambiguous. It found a clean separating structure that already exists in our engineered features:
* “providers with no robust events” vs
* “providers with at least one robust event”

That matches our cluster profile medians:
* `cluster_0` median `n_anom_rows_robust` = 0
* `cluster_1` median `n_anom_rows_robust` = 1

And it visually justifies the labels we chose.

---

### 5) What we can say in an executive summary, precisely

> “Among eligible providers, the clustering cleanly splits into a large group with **zero robust tail events**, and a smaller group with **one or more robust tail events**.”

> “Within the elevated-anomaly group, the burden is usually small (most have 1–2 events), but there is a long tail of providers with higher repeat-event counts, which are natural candidates for deeper review.”

# TIER.6.B2) Distribution overlap (eligible only): Boxplot of `anom_rate_pct_robust` by cluster

In [ ]:
# ============================================================
# TIER.6.B2) Distribution overlap (eligible only): Boxplot of anom_rate_pct_robust by cluster
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

score = provider_scorecard_v1.copy()
eligible = score[score["cluster_label_v1"].astype(str).str.startswith("cluster_")].copy()

groups = []
labels = []
for cl, g in eligible.groupby("cluster_label_v1", dropna=False):
    groups.append(pd.to_numeric(g["anom_rate_pct_robust"], errors="coerce").fillna(0).to_numpy())
    labels.append(cl)

fig = plt.figure(figsize=(10, 6))
ax = fig.add_subplot(111)
ax.boxplot(groups, tick_labels=labels, showfliers=False)
ax.set_title("Eligible providers: robust anomaly-rate distribution by cluster")
ax.set_ylabel("anom_rate_pct_robust")
plt.tight_layout()
plt.show()

### What’s being plotted

Each point that went into this plot is one provider (eligible providers only).
The y-axis is `anom_rate_pct_robust`, defined as:

> For a given provider,  
> `anom_rate_pct_robust = 100 × ( # of provider rows flagged as robust extreme events / # of provider rows )`.

So if a provider has 200 rows (`n_rows_rob = 200`) and 2 robust extreme events (`n_anom_rows_robust = 2`), their rate is **1.0%**.

The boxplot summarizes the distribution of that provider-level rate within each cluster:

* **Orange line** = median provider rate
* **Box** = middle 50% (25th to 75th percentile)
* **Whiskers** = typical range beyond the box (matplotlib’s default is usually up to ~1.5×IQR, so not “min/max” unless there are no outliers)

---

### What it proves, visually

#### `cluster_0` is a true “zero-rate” bucket

* The entire box for `cluster_0` is basically sitting at **0**.
* That matches our profile table: `cluster_0` has median `n_anom_rows_robust = 0` and median rate = 0.

**Interpretation:**
For essentially all providers in `cluster_0`, **robust extreme events never happen** under our strict gating.

This is the cleanest possible separation we can get in a tiering story.

#### `cluster_1` is the “repeat-offender rate” bucket

`cluster_1`’s box is clearly above 0:

* **Median** is around **~1.0%** (this matches our cluster card: median `anom_rate_pct_robust = 1.013%`).
* The **IQR** (middle 50% of providers) looks roughly around **~0.6% to ~1.6%**.
* The **upper whisker** reaches roughly **~3.1–3.2%**.

**Interpretation:**
A typical provider in `cluster_1` has about **1 in 100 rows** flagged as a robust extreme event. Many are in the “sub-2%” range, and the heavier tail providers reach ~3% of rows flagged.

---

### Why this plot is powerful in our narrative

The ECDF showed separation in **counts** (`n_anom_rows_robust`).
This boxplot shows separation in **rates**, which is more “fair” across providers with different row volumes.

Together, they make a strong claim:

* **`cluster_0`:** eligible providers with **no robust anomaly burden**.
* **`cluster_1`:** eligible providers with a **systematic anomaly burden**, not just one-off noise.

---

### How to explain `cluster_1`’s median of ~1%

Remember what a “robust extreme event” is in our strict definition:

* directionality (over-expected),
* high-confidence gate (`is_high_conf`),
* within-slice top 1% (`log_oe_pct_in_slice >= 0.99`),
* slice validity (`slice_n >= 50`).

So the boxplot is essentially saying:

> “Among eligible providers, this cluster contains the ones who repeatedly land in the within-peer top 1% tail, under a strict confidence gate.”

That’s exactly aligned with our labels:

* **`cluster_0`:** Typical cost behavior (no robust tail events)
* **`cluster_1`:** Elevated anomaly burden (robust tail events present)

------------------
------------------
------------------
------------------
------------------


# Provider Tiering Notebook (v1) — End-to-End Summary

This notebook builds a **provider-level tiering layer** on top of the scored CMS universe and the anomaly signals produced by the expected-cost model. The output is a **single, stable provider scorecard** plus a **cluster-based segmentation** of “eligible” providers into interpretable groups, with artifacts frozen to disk for reproducibility.

---

## What this notebook produces (final artifacts)

### Core provider table (single source for downstream modules)
- **`provider_scorecard_v1_clustered`** (one row per `(Rndrng_NPI, provider_type, state)`)
  - contains provider-level cost-performance stats, repeat-offender metrics, shock metrics, scale metrics, and cluster assignments
  - includes:
    - `cluster_id_v1` (numeric cluster id for eligible providers)
    - `cluster_label_v1` (e.g., `cluster_0`, `cluster_1`, or `ineligible`)
    - `cluster_definition_v1` (human-readable label)
    - `dist_to_centroid_v1` (distance in clustering feature space, smaller means “more typical within cluster”)

### Clustering artifacts (audit + reproducibility)
- **`tier_clusters_v1`** (eligible providers only, with `cluster_id_v1` + distance)
- **`k_sweep_v1`** (k, inertia, silhouette across the tested range)
- **Cluster profiling outputs**
  - `cluster_profiles_v1_compact` (cluster-level medians + p95 on raw features; includes human labels)

All outputs are written under `artifacts/provider_tiering/run_{timestamp}/` with:
- `params__{ts}.json` + `params__{ts}.md`
- `manifest__{ts}.json`

---

## Design principles (what makes this defensible)

1. **Stable grain**
   - Everything rolls up to provider grain: **`(Rndrng_NPI, provider_type, state)`**.
   - This prevents mixing selection logic with feature logic and makes the outputs easy to join into UI/query workflows.

2. **Separation of concerns**
   - We explicitly separate:
     - scored row-level universe → minimal anomaly feature layer → provider summaries → provider scorecard → clustering features → clustering → cluster profiling.
   - Each step has a clear purpose and can be audited independently.

3. **Eligibility gate**
   - We do not cluster providers with insufficient data.
   - “Ineligible” is a first-class bucket, not a silent failure case.
   - This prevents nonsense clusters driven by tiny denominators.

4. **Value-focused clustering**
   - We explicitly avoid “size clustering”.
   - Volume features are used for eligibility and reporting, not as distance drivers.
   - We document this with a distance-domination justification.

---

# Step-by-step workflow

## TIER.0) Load scored universe (single source of truth)

**Input**: `eval_scored_DG_V3.parquet` (the scored universe at row grain)

**Row grain (important)**  
Each row corresponds to the scoring unit used in the engine (same universe used in anomaly surfacing), at the level of:
- `(Rndrng_NPI, HCPCS_Cd, Place_Of_Srvc, Year)`  
plus model outputs and supporting features.

**What we validate immediately**
- shape and schema
- duplicate key rows
- sanity distributions for:
  - `expected_cost`
  - `observed_cost`
  - `residual`
  - `log_oe`
  - `oe_ratio`
- extreme tail diagnostics (e.g., how many rows have `oe_ratio > 10/50/100/1000`, and whether those tails are driven by near-zero denominators)

**Output**
- `eval_base`: the loaded scored universe used as the base for tiering.

---

## TIER.1) Minimal anomaly feature layer for tiering

**Goal**: Add only the anomaly features needed for provider aggregation (not the full anomaly notebook).

### 1) Confidence flag: `is_high_conf`
We define `is_high_conf` as a strict, auditable rule:
- support tier in `{high, medium_high}`
- and `services >= MIN_SERVICES`
- and `benes >= MIN_BENES`

This gives a stable definition of “rows with enough support to trust”.

### 2) Slice-aware features (peer comparability)
We compute slice features within:
- `slice_cols = (HCPCS_Cd, Year)`

We create:
- `log_oe_pct_in_slice`: percentile rank of `log_oe` inside each `(HCPCS_Cd, Year)` slice
- `slice_n`: number of rows in the slice (peer group size)

**Why**
- `log_oe_pct_in_slice` makes “extreme” meaningfully relative to peers.
- `slice_n` lets us gate out tiny slices so “top 1%” is not meaningless.

**Output**
- `eval_tier`: `eval_base` plus:
  - `is_high_conf`
  - `log_oe_pct_in_slice`
  - `slice_n`

---

## TIER.2) Provider summaries (repeat offenders + shocks), no TOP_N trimming

**Goal**: Compute provider-level summaries for *all providers*, not just top worklists.

**Shared knobs**
- `MIN_SLICE_N = 50` (slice validity gate)
- denominator hygiene (optional but recommended):
  - `USE_MIN_EXPECTED_FILTER = True`
  - `MIN_EXPECTED = 1e-2`
- shock severity quantile:
  - `LOG_OE_Q = 0.995`
  - `log_oe_severity_cut = quantile(log_oe, q=0.995)` computed on `BASE = rows with (log_oe>0 & residual>0)` after denominator hygiene

### A) Robust repeat-offender event flag (size-aware)
We define a robust “extreme event” row flag:

`is_row_anomalous_robust_sizeaware` is True when:
- `log_oe > 0` and `residual > 0` (directional)
- `is_high_conf` is True (strict confidence gate for tiering)
- `log_oe_pct_in_slice >= 0.99` (top 1% within peers)
- `slice_n >= MIN_SLICE_N` (enough peers)

Then we aggregate per provider `(Rndrng_NPI, provider_type, state)`:
- frequency: `n_anom_rows_robust`
- rate: `anom_rate_pct_robust`
- breadth: `n_unique_codes`, `n_unique_years`
- cost-performance stats: `median_log_oe`, `median_residual`, `p75/p90 log_oe`, `p75/p90 residual`
- scale: `total_services`, `total_benes`
- confidence density: `pct_high_conf_rows`

This produces:
- `provider_summary_robust_sizeaware_all`

### B) Shock event flag (magnitude-aware + size-aware)
We define a “shock event” as a stricter tail:
- still directional and slice-relative
- plus a global severity cut

`is_row_anomalous_mag_sizeaware` is True when:
- `log_oe > 0` and `residual > 0`
- `is_high_conf` is True
- `log_oe_pct_in_slice >= 0.99`
- `log_oe >= log_oe_severity_cut` (global top tail by definition)
- `slice_n >= MIN_SLICE_N`

Provider shock metrics are computed on anomalous rows only:
- `n_anom_rows_mag`
- `max_log_oe_mag`, `p95_log_oe_mag` (masked to shock events)
- plus scale fields for reporting

This produces:
- `provider_summary_mag_sizeaware_all`

**Outputs**
- `provider_summary_robust_sizeaware_all` (all providers)
- `provider_summary_mag_sizeaware_all` (all providers)
- printed headline counts:
  - number of providers with ≥1 robust event
  - number of providers with ≥1 shock event

---

## TIER.3) Build provider_scorecard_v1 + freeze

**Goal**: Create the one-table “provider representation” used by all downstream modules.

We merge:
- robust provider summary (repeat offender features)
- magnitude provider summary (shock features)

Join key:
- `(Rndrng_NPI, provider_type, state)`

We fill missing shock metrics for providers without shock events:
- set shock counters and severities to 0 (by construction)

We add helpful boolean flags:
- `has_any_robust_event`
- `has_any_shock_event`

**Output**
- `provider_scorecard_v1` (one row per provider grain)
- frozen to:
  - parquet + csv
  - params + manifest

This is the stable provider-level table for tiering, explainability, and UI ranking.

---

## TIER.4) Build tier_features_v1 (clean + scaled + defensible)

**Goal**: Prepare a defensible clustering feature matrix with:
- a small feature set
- tail control (winsorization)
- robust scaling
- explicit eligibility gating

### A) Eligibility gate
We define providers eligible for clustering if they have enough evidence:
- `n_rows_rob >= MIN_N_ROWS_FOR_CLUSTER`
- `total_services_rob >= MIN_SERVICES_FOR_CLUSTER`

This yields:
- `is_cluster_eligible_v1` (True/False)

### B) Raw features (initial list)
We initially consider features spanning:
- cost-performance
- repeat offender behavior
- shock severity
- scale/confidence

### C) Tail handling: winsorization
For heavy-tailed variables, we clip them to `[p01, p99]`:
- reduces domination from extreme outliers
- keeps ordering and interpretability

### D) Robust scaling
We robust-scale each feature:
- `(x - median) / IQR` (fallback to std if IQR=0)
This makes features comparable in Euclidean distance.

**Outputs**
- `tier_features_v1` (all providers with raw + winsor + scaled columns)
- `tier_features_v1_clustering` (eligible-only subset with scaled cols)

---

## TIER.4.JUSTIFY) Why we exclude some features from clustering (TIER.4 -> Option A)

**Goal**: Make the decision to exclude certain features explicit and evidence-based.

We run three diagnostics on the eligible set:

1) **Redundancy check**
- look for high correlations among scaled features (e.g., `abs(corr) >= 0.85`)
- identifies redundant features that do not add unique signal

2) **Distance domination check**
- compute each feature’s share of total squared Euclidean distance mass across many random provider pairs
- if a feature contributes ~all distance, clustering becomes “that feature only”

3) **Size-tie check**
- correlate features with `total_services_rob_rs`
- identifies features that are effectively proxies for size and would yield “size clusters”

**Key result**
- volume features (especially `total_services_rob`) dominate distance if included.
- some breadth/volume proxies correlate strongly with size.

This justifies a value-focused option where volume is used for eligibility, not distance.

---

## TIER.4.a / TIER.4.b) Value-focused clustering feature set (locked)

**Goal**: Define the final clustering columns explicitly and freeze the decision.

We build a clustering matrix using only robust-scaled, value-focused columns:
- `median_log_oe_rob_rs`
- `p90_log_oe_rs`
- `n_anom_rows_robust_rs`
- `anom_rate_pct_robust_rs`
- `p95_log_oe_mag_rs`
- `pct_high_conf_rows_rs`

This becomes:
- `CLUSTER_COLS_LOCKED`
- `tier_features_v1_clustering_LOCKED`

---

## TIER.5) Cluster eligible providers + attach cluster labels + freeze artifacts

**Goal**: Run KMeans on the eligible set and attach cluster outputs to the scorecard.

### A) Choose K (small sweep)
We sweep K from `K_MIN..K_MAX` and compute:
- **inertia** (within-cluster SSE; lower is better)
- **silhouette** (separation/compactness; higher is better)

We pick `K_CHOSEN` by maximum silhouette (default rule for v1).

### B) Fit final KMeans
We fit KMeans on the eligible feature matrix:
- cluster assignments (`cluster_id_v1`)
- compute `dist_to_centroid_v1` for each provider (Euclidean distance in scaled space)

### C) Attach clusters back to:
- eligible-only table (`tier_clusters_v1`)
- full provider scorecard (`provider_scorecard_v1`, with `ineligible` preserved as no cluster)

### D) Freeze clustering artifacts
We write:
- `tier_clusters_v1`
- `provider_scorecard_v1_labeled`
- `k_sweep_v1`
plus params and manifest.

---

## TIER.6) Cluster profiling (cluster-only) + freeze artifacts

**Goal**: Turn clusters into interpretable segments with human-readable definitions.

### A) Cluster profile tables (eligible clusters only)
We compute cluster-level summary stats on:
- the raw versions of LOCKED features
- plus reporting-only columns (volume, breadth, scores)

We output:
- `cluster_profiles_v1` (full multi-agg table)
- `cluster_profiles_v1_compact` (medians + p95 only)

### B) Cluster ordering score (reference-only)
We compute `tier_order_score_v1` (reference ordering metric):
- take cluster medians on a small “worse value” set:
  - `median_log_oe_rob`, `p90_log_oe`, `n_anom_rows_robust`, `anom_rate_pct_robust`
- z-score these across clusters (to avoid unit dominance)
- sum the z-scores to create a single ordering number
  - lower = “less anomaly burden”
  - higher = “more anomaly burden”

This is used only to **order clusters consistently**. It is not presented as quality truth.

### C) Attach human cluster definitions
We assign:
- `cluster_definition_v1` for each cluster label, for example:
  - `cluster_0`: Typical cost behavior (no robust tail events)
  - `cluster_1`: Elevated anomaly burden (robust tail events present)

We keep “Tier_A/Tier_B” as a UI-ordering concept only, not as a semantic label.

### D) Freeze profiling outputs
We write:
- `cluster_profiles_v1_compact`
- `provider_scorecard_v1_clustered`
plus params and manifest.

---

## TIER.6.A) One-slide “cluster label card” (executive summary table)

We produce a compact cluster definition table for eligible providers:

For each cluster:
- `n_providers`
- median `n_anom_rows_robust`
- median `anom_rate_pct_robust`
- median `p90_log_oe` (simple, interpretable “upper-tail typical”)
- `% with any shock event` (share with `n_anom_rows_mag > 0`)
- median `total_services_rob`
- `cluster_definition_v1`

This is designed to be “drop into a slide” with minimal explanation overhead.

---

## TIER.6.B) Distribution overlap plots (visual proof)

We include a simple plot that proves the clusters are meaningfully different:

### B1) ECDF of `n_anom_rows_robust` by cluster (eligible only)
- shows cluster separation on the repeat-offender count
- visually demonstrates that `cluster_0` is concentrated at 0 events, while `cluster_1` has nonzero mass at 1+ events.

### B2) Boxplot of `anom_rate_pct_robust` by cluster (eligible only)
- shows cluster separation on anomaly rate (frequency normalized)
- reinforces the “monitoring bucket” interpretation for the elevated-burden cluster.

---

# What the tiering v1 means (and what it does not mean)

## What it IS
A defensible segmentation of **eligible providers** into:
- **Typical cost behavior** (no robust tail events under strict gating)
- **Elevated anomaly burden** (repeat-offender tail events present)
and a third explicit bucket:
- **Ineligible** (insufficient evidence to cluster)

This is already valuable because it turns millions of rows into an operationally usable provider segmentation:
- “stable / typical”
- “monitor / investigate”
- “insufficient evidence”

## What it is NOT (yet)
- not a clinical quality tier
- not a “Gold/Silver/Bronze” value tier in the payer sense
- not a fraud label
- not a final decision tool

It is a tiering layer grounded in model-based benchmarking and defensible anomaly definitions.

---

# How this connects to the provider benchmarking engine UI

This tiering layer plugs directly into product logic:

Given a query context (geo + provider type + service):
- show provider rows with:
  - eligibility status
  - cluster definition
  - scorecard metrics (cost-performance + anomaly burden)
  - drill-down links to:
    - row-level worklists (extreme rows)
    - provider-level anomaly summaries (repeat offenders vs shocks)

This supports:
- ranking
- filtering
- explainability
- monitoring workflows

---

# Reproducibility and audit trail (what to keep frozen)

For every run we freeze:
- the core output table(s) in parquet + csv
- params capturing:
  - confidence thresholds (MIN_SERVICES, MIN_BENES, tiers)
  - slice definition and MIN_SLICE_N
  - MIN_EXPECTED filter settings
  - LOG_OE_Q and severity cut value
  - eligibility gate thresholds
  - clustering columns (LOCKED)
  - k sweep range, random_state, n_init, chosen k
- manifest listing written files

This ensures every tier label can be traced back to:
- the scored universe
- the feature definitions
- the clustering configuration

---

# Next steps (if continuing)

Recommended next steps after this notebook:
1. Add “cluster label cards” and distribution plot(s) into the executive summary deck.
2. Build ranking logic in the product layer (cluster + expected/observed + confidence).
3. Optionally explore:
   - alternative clustering methods (GMM, hierarchical)
   - stability checks (bootstrap, reruns)
   - additional segmentation layers (within-cluster prioritization using `dist_to_centroid_v1`, shock flags, and business thresholds).